# 🧱 Base analítica: do lake à tabela de modelagem

Este notebook monta a **tabela analítica** que alimentará os modelos: cada linha é um aluno avaliado, com a variável resposta (alfabetizado ou não) e o contexto do seu município. É a primeira etapa do projeto e a fundação de todas as seguintes, porque define a unidade de predição, as variáveis disponíveis e a estratégia que impede o vazamento de dados.

> **Convenção de trabalho:** o desenvolvimento acontece neste notebook, célula a célula, com os resultados salvos. Ao final da etapa, o código estável é promovido para `src/preprocessing/prod_01_base_analitica.py`.

**Decisões que governam este notebook** (ver [diário de decisões](../docs/decisoes.md)):
- **D-001, grão de modelagem:** a linha vem da camada Silver (o aluno), o contexto vem da camada Gold e das tabelas municipais (o território), sempre defasado no tempo;
- **D-002, dependência entre as fases:** este projeto verifica o contrato do lake construído na fase anterior, sem reexecutar aquela pipeline.

---

**Estratégia deste notebook:**

```
verificar o       ->  isolar a          ->  montar o          ->  integrar e   ->  desenhar o    ->  gravar a
lake (pré-voo)        população            contexto              auditar          split             base
Seção 1               Seção 2              defasado (3)          Seções 4 e 5     Seção 6           Seção 7
```

O princípio é o mesmo da fase anterior: cada passo é conferido contra um número esperado antes do seguinte, e nada é gravado sem verificação.

## 1. Setup e verificação de pré-requisitos

**Passos desta seção:** (1.1) configurar o acesso ao data lake da fase anterior; (1.2) verificar se as camadas exigidas existem e têm as colunas esperadas.

🎓 **Conceito, contrato entre pipelines:** a pipeline de dados da fase anterior é *upstream*; este projeto de machine learning é *downstream*. Quem está a jusante **verifica** se o insumo existe no formato esperado, e não reexecuta a pipeline de cima: reexecutar acoplaria os dois repositórios e duplicaria responsabilidade. Se algo faltar, a execução para com uma mensagem que orienta o que rodar antes (decisão D-002).

> 📌 **Nota para reprodução:** este notebook lê `config/config.json`, que não é versionado. Crie o seu a partir de `config/config.example.json`, apontando para o projeto e o bucket onde a pipeline da fase anterior foi executada.

In [197]:
# --- 1.1 Configuração e acesso ao data lake ---
import json
from pathlib import Path

import pandas as pd
import pydata_google_auth
from google.cloud import storage

CFG = json.loads(Path("../config/config.json").read_text(encoding="utf-8"))
PROJETO_GCP = CFG["projeto_gcp"]
BUCKET_LAKE = CFG["bucket_lake"]

ESCOPOS = ["https://www.googleapis.com/auth/cloud-platform"]
credenciais = pydata_google_auth.get_user_credentials(ESCOPOS)
credenciais = credenciais.with_quota_project(PROJETO_GCP)
cliente_storage = storage.Client(project=PROJETO_GCP, credentials=credenciais)

def garantir_credencial(forcar: bool = False) -> None:
    """Renova o token de acesso e limpa o cache do gcsfs.

    O token de usuário expira em cerca de uma hora. Em sessões longas de
    notebook, qualquer leitura ou gravação no lake feita depois disso
    falha com erro 401. Esta função deve ser chamada antes de cada acesso;
    o parâmetro forcar permite renovar mesmo quando o token ainda parece
    válido, situação em que o gcsfs pode manter em cache uma instância
    autenticada com o token anterior.
    """
    import google.auth.transport.requests
    import gcsfs
    if forcar or not credenciais.valid:
        credenciais.refresh(google.auth.transport.requests.Request())
        gcsfs.GCSFileSystem.clear_instance_cache()

def ultima_particao(area: str, tabela: str) -> str | None:
    """Partição mais recente de uma tabela do lake, ou None se não existir."""
    particoes = sorted({
        b.name.split("/")[2]
        for b in cliente_storage.list_blobs(BUCKET_LAKE, prefix=f"{area}/{tabela}/")
        if len(b.name.split("/")) > 2
    })
    return particoes[-1] if particoes else None

def ler_lake(area: str, tabela: str, **kwargs) -> pd.DataFrame:
    """Lê a partição mais recente de uma tabela do lake."""
    garantir_credencial()
    particao = ultima_particao(area, tabela)
    if particao is None:
        raise FileNotFoundError(
            f"Tabela '{tabela}' não encontrada em {area}/. "
            "Execute antes a pipeline da fase anterior."
        )
    caminho = f"gs://{BUCKET_LAKE}/{area}/{tabela}/{particao}/{tabela}.parquet"
    return pd.read_parquet(caminho, storage_options={"token": credenciais},
                           **kwargs)

print(f"Lake: gs://{BUCKET_LAKE}/")
print(f"Projeto GCP: {PROJETO_GCP}")
print("Setup ok")

Lake: gs://tech-challenge-fase2-lake-rm373453/
Projeto GCP: tech-challenge-fase2
Setup ok


In [198]:
# --- 1.2 Verificação de pré-requisitos (pré-voo) ---
# Contrato: tabelas e colunas que este projeto exige da fase anterior.
CONTRATO = {
    ("silver", "alunos"): ["ano", "id_municipio", "rede_nome", "presente",
                           "alfabetizado", "proficiencia", "peso_aluno"],
    ("silver", "municipio"): ["ano", "id_municipio", "rede", "nome",
                              "sigla_uf", "nome_regiao",
                              "taxa_alfabetizacao", "media_portugues"],
    ("silver", "metas_municipio"): ["ano_referencia", "id_municipio",
                                    "ano_meta", "meta_taxa"],
    ("gold", "indicador_municipio"): ["ano", "id_municipio", "taxa",
                                      "percentual_participacao",
                                      "meta_taxa", "origem"],
}

print("Verificação de pré-requisitos no lake:")
print()
pendencias = []
for (area, tabela), colunas in CONTRATO.items():
    particao = ultima_particao(area, tabela)
    if particao is None:
        print(f"  {area}/{tabela:<22} AUSENTE")
        pendencias.append(f"{area}/{tabela}")
        continue
    amostra = ler_lake(area, tabela, columns=colunas[:1])
    disponiveis = set(pd.read_parquet(
        f"gs://{BUCKET_LAKE}/{area}/{tabela}/{particao}/{tabela}.parquet",
        storage_options={"token": credenciais}).columns)
    faltantes = [c for c in colunas if c not in disponiveis]
    status = "OK" if not faltantes else f"COLUNAS FALTANTES: {faltantes}"
    if faltantes:
        pendencias.append(f"{area}/{tabela}: {faltantes}")
    print(f"  {area}/{tabela:<22} {particao}  {len(amostra):>9,} linhas  {status}")

print()
if pendencias:
    raise RuntimeError(
        "Pré-requisitos não atendidos: " + "; ".join(pendencias) + ".\n"
        "Execute a pipeline da fase anterior (repositório "
        "Tech_Challenge_RM373453_pipeline_alfabetizacao, passos 4 a 8 do "
        "Como Executar) antes de prosseguir."
    )
print("Contrato atendido: o lake tem o que este projeto precisa.")

Verificação de pré-requisitos no lake:



  silver/alunos                 data_processamento=2026-07-11  3,866,814 linhas  OK
  silver/municipio              data_processamento=2026-07-11     23,995 linhas  OK
  silver/metas_municipio        data_processamento=2026-07-11     74,928 linhas  OK
  gold/indicador_municipio    data_processamento=2026-07-12     11,629 linhas  OK

Contrato atendido: o lake tem o que este projeto precisa.


## 2. População de modelagem e variável resposta

**Passos desta seção:** (2.1) carregar os alunos da camada Silver e isolar a população modelável; (2.2) examinar a variável resposta e seu balanceamento.

🎓 **Conceito, população modelável** (decisão D-001): o modelo aprende com quem tem resultado observado. Alunos ausentes na avaliação não têm proficiência registrada e, portanto, não têm variável resposta: eles permanecem no lake, com a flag de presença criada na fase anterior, mas ficam fora do treinamento. Essa exclusão é uma premissa declarada, não um descarte silencioso, e volta como limitação do projeto: o modelo prevê a alfabetização de quem realiza a prova.

⚠️ **Achado do reconhecimento do lake:** o identificador de aluno se repete entre os ciclos de 2023 e 2024 em 86,7% dos casos, embora a avaliação seja aplicada a coortes diferentes (alunos do 2º ano de cada ano). O identificador é, portanto, uma máscara reutilizada, e não permite acompanhar a mesma criança ao longo do tempo. Isso exclui qualquer variável de trajetória individual e reforça que o identificador não entra no modelo.

In [199]:
# --- 2.1 Carregar alunos e isolar a população modelável ---
df_alunos = ler_lake(
    "silver", "alunos",
    columns=["ano", "id_municipio", "rede_nome", "presente",
             "alfabetizado", "proficiencia", "peso_aluno"],
)
print(f"Alunos na camada Silver: {len(df_alunos):,}")

# A população modelável: presentes na avaliação (D-001)
df_pop = df_alunos[df_alunos["presente"]].copy()
print(f"Presentes (com resultado observado): {len(df_pop):,} "
      f"({len(df_pop) / len(df_alunos):.1%})")
print(f"Ausentes (fora do treinamento):      "
      f"{len(df_alunos) - len(df_pop):,}")
print()

# Variável resposta: alfabetizado (1) ou não (0)
df_pop["alvo"] = (df_pop["alfabetizado"].astype(str) == "1").astype(int)

print("Distribuição por ciclo:")
print(df_pop.groupby("ano")["alvo"]
      .agg(alunos="size", alfabetizados="sum",
           taxa=lambda s: f"{100 * s.mean():.1f}%").to_string())

Alunos na camada Silver: 3,866,814
Presentes (com resultado observado): 3,354,661 (86.8%)
Ausentes (fora do treinamento):      512,153

Distribuição por ciclo:
       alunos  alfabetizados   taxa
ano                                
2023  1502809         877427  58.4%
2024  1851852        1107119  59.8%


In [200]:
# --- 2.2 Conferir a variável resposta contra o dado oficial ---
# A taxa da população modelável é a taxa NÃO ponderada; a oficial usa o
# peso amostral. As duas devem ficar próximas, e a diferença entre elas
# já antecipa a decisão pendente sobre o uso do peso no treinamento.

for ano in sorted(df_pop["ano"].unique()):
    recorte = df_pop[df_pop["ano"] == ano]
    simples = 100 * recorte["alvo"].mean()
    ponderada = (100 * (recorte["peso_aluno"] * recorte["alvo"]).sum()
                 / recorte["peso_aluno"].sum())
    print(f"{ano}:  simples {simples:.1f}%   ponderada {ponderada:.1f}%   "
          f"diferença {ponderada - simples:+.1f} pp")

print()
print("Referência oficial da rede pública (INEP): 55,9% em 2023 e 59,2% em 2024.")
print("A população acima inclui a rede privada, o que eleva ambas as taxas.")
print()
print("Distribuição por rede (todos os ciclos):")
print(df_pop.groupby("rede_nome")["alvo"]
      .agg(alunos="size", taxa=lambda s: f"{100 * s.mean():.1f}%").to_string())

2023:  simples 58.4%   ponderada 57.5%   diferença -0.9 pp
2024:  simples 59.8%   ponderada 59.2%   diferença -0.6 pp

Referência oficial da rede pública (INEP): 55,9% em 2023 e 59,2% em 2024.
A população acima inclui a rede privada, o que eleva ambas as taxas.

Distribuição por rede (todos os ciclos):
            alunos   taxa
rede_nome                
Estadual    372596  62.1%
Municipal  2982041  58.8%
Privada         24  66.7%


### 2.3 Dois achados que orientam as próximas decisões

- **O peso amostral reproduz o número oficial.** A taxa ponderada de 2024 (59,2%) coincide com a taxa divulgada pelo INEP para a rede pública naquele ciclo, enquanto a taxa simples fica 0,6 ponto acima. O peso, calibrado pelo instituto, corrige a sub-representação de certos perfis na amostra: é a evidência que sustentará a decisão sobre usá-lo como ponderação no treinamento.
- **A rede privada é residual nesta base:** 24 alunos entre 3,35 milhões. Não constitui uma categoria com massa estatística; será tratada de forma explícita na preparação, para não gerar uma classe rara sem significado no modelo.

## 3. Contexto defasado: da rede do aluno e do seu município

**Passos desta seção:** (3.1) montar o retrato da rede de ensino de cada município no ciclo anterior; (3.2) acrescentar o retrato do município como um todo e as variáveis conhecidas antes da avaliação.

🎓 **Conceito, o que o modelo pode saber** (*informação ex-ante × ex-post*): a regra que separa uma variável legítima de um vazamento não é o assunto dela, é o **momento em que ela passa a existir**. No instante da predição já se conhece a situação do território no ciclo anterior e a meta pactuada para o ciclo corrente; ainda não se conhece o resultado do próprio ciclo.

| Variável | Quando passa a existir | Entra no modelo |
|---|---|---|
| Desempenho da rede e do município no ciclo **anterior** | antes da avaliação | ✅ sim |
| **Meta** pactuada para o ciclo corrente | antes da avaliação (é um pacto prévio) | ✅ sim |
| Taxa do município no ciclo **corrente** | depois da avaliação, e calculada **com o próprio aluno** | ❌ não |

O caso da meta merece destaque: ela se refere ao ciclo que se quer prever, mas é **conhecida de antemão**, porque foi pactuada entre os entes federativos antes da aplicação da prova. Não é vazamento; é exatamente a informação que um gestor teria em mãos ao tentar antecipar o resultado.

🎓 **Conceito, o grão do contexto:** o aluno pertence a uma **rede** dentro de um **município**, e as duas camadas informam coisas diferentes. A rede estadual e a rede municipal de uma mesma cidade divergem mais do que se imagina: nos 1.083 municípios em que ambas foram medidas em 2023, a diferença entre elas tem desvio padrão de 19,5 pontos percentuais e supera 10 pontos em 58% dos casos. Usar apenas o agregado das duas descartaria essa variação e trataria alunos de realidades distintas como se vivessem a mesma. Por isso o contexto é montado em dois níveis:

- **rede do aluno**, o desempenho da rede que efetivamente o atende;
- **município**, o clima educacional do território como um todo, com participação e porte;
- e a **diferença entre os dois**, que posiciona a rede do aluno acima ou abaixo do seu município.

📌 **Consequência do desenho temporal:** como a camada Silver cobre 2023 e 2024, apenas o ciclo de **2024** reúne aluno e contexto anterior. O ciclo de 2023 **não entra como linha, e sim como contexto**: o retrato daquele ano vira coluna em todas as observações. Isso evita dois erros opostos, o de misturar ciclos deixando metade das linhas sem contexto, e o de usar o contexto do próprio ciclo, que seria vazamento.

In [201]:
# --- 3.1 Retrato da rede de ensino no ciclo anterior ---
CICLO_ALVO = 2024
CICLO_ANTERIOR = CICLO_ALVO - 1

mun = ler_lake("silver", "municipio",
               columns=["ano", "id_municipio", "rede", "rede_nome",
                        "taxa_alfabetizacao", "media_portugues",
                        "sigla_uf", "nome_regiao"])

# Contexto no grão da rede: a rede que atende o aluno (Estadual ou Municipal)
ctx_rede = mun.loc[
    (mun["ano"] == CICLO_ANTERIOR) & (mun["rede"].astype(str).isin(["2", "3"])),
    ["id_municipio", "rede_nome", "taxa_alfabetizacao", "media_portugues"],
].rename(columns={
    "taxa_alfabetizacao": "rede_taxa_ant",
    "media_portugues": "rede_media_portugues_ant",
})

print(f"Retratos de rede em {CICLO_ANTERIOR}: {len(ctx_rede):,}")
print(ctx_rede.groupby("rede_nome", observed=True)
      .agg(municipios=("id_municipio", "nunique"),
           taxa_media=("rede_taxa_ant", lambda s: f"{s.mean():.1f}%"))
      .to_string())
print()

# Quanto as redes divergem dentro do mesmo município
comparativo = ctx_rede.pivot_table(index="id_municipio", columns="rede_nome",
                                   values="rede_taxa_ant", observed=True)
ambas = comparativo.dropna()
diferenca = ambas["Estadual"] - ambas["Municipal"]
print(f"Municípios com as duas redes medidas: {len(ambas):,}")
print(f"Diferença Estadual - Municipal: média {diferenca.mean():+.1f} pp, "
      f"desvio {diferenca.std():.1f} pp")
print(f"Acima de 10 pp de diferença: {100 * (diferenca.abs() > 10).mean():.0f}% "
      f"dos municípios")

Retratos de rede em 2023: 6,597
           municipios taxa_media
rede_nome                       
Estadual         1149      63.8%
Municipal        5448      60.3%

Municípios com as duas redes medidas: 1,083
Diferença Estadual - Municipal: média +3.7 pp, desvio 19.5 pp
Acima de 10 pp de diferença: 58% dos municípios


In [ ]:
# --- 3.2 Retrato do município e variáveis conhecidas antes da avaliação ---
# Município como um todo (rede pública), vindo da camada Gold
gold = ler_lake("gold", "indicador_municipio")
ctx_mun = gold.loc[
    (gold["ano"] == CICLO_ANTERIOR) & (gold["origem"] == "oficial_inep"),
    ["id_municipio", "taxa", "percentual_participacao", "taxa_ajustada",
     "alunos_presentes"],
].rename(columns={
    "taxa": "mun_taxa_ant",
    "percentual_participacao": "mun_participacao_ant",
    "taxa_ajustada": "mun_taxa_ajustada_ant",
    "alunos_presentes": "mun_alunos_ant",
})

# O território (UF e região) não vem daqui: a tabela municipal do ciclo
# anterior só cobre os municípios avaliados naquele ano, e deixava cerca de
# 10% dos alunos sem UF. Ele vem do Censo Demográfico de 2022 (seção 4.7),
# que cobre todos os municípios do país.

# Meta pactuada para o ciclo alvo: informação ex-ante
metas = gold.loc[
    (gold["ano"] == CICLO_ALVO) & (gold["origem"] == "oficial_inep"),
    ["id_municipio", "meta_taxa"],
].rename(columns={"meta_taxa": "mun_meta_ciclo"})
ctx_mun = ctx_mun.merge(metas, on="id_municipio", how="left")
ctx_mun["mun_gap_meta"] = ctx_mun["mun_meta_ciclo"] - ctx_mun["mun_taxa_ant"]

# Espaço reservado para o enriquecimento externo (etapa 3 do plano):
# fontes municipais novas entram aqui, por join em id_municipio.

print(f"Contexto municipal: {len(ctx_mun):,} municípios, "
      f"{len(ctx_mun.columns) - 1} variáveis")
print()
print("Preenchimento das variáveis municipais:")
print(pd.DataFrame({
    "% preenchido": (100 * ctx_mun.notna().mean()).round(1)
}).drop(index="id_municipio").to_string())

### 3.3 Benchmark estadual: comparar a rede com os seus pares

🎓 **Conceito, posição relativa ao grupo de pares:** o valor absoluto de um indicador diz pouco sem referência. Uma rede municipal com 60% de alfabetização representa uma situação boa em um estado cuja mediana é 50%, e ruim em outro cuja mediana é 70%. O que informa é a **posição relativa**, e os pares corretos para comparação são as redes do **mesmo tipo**, no **mesmo estado**, porque compartilham política estadual, contexto socioeconômico e regime de colaboração entre os entes.

Duas variáveis nascem daí: o benchmark em si (a mediana da rede na unidade da federação) e a distância da rede do aluno até ele. A mediana é preferida à média por ser robusta a municípios atípicos, comuns em estados com poucas redes medidas.

📌 **Sem vazamento:** o benchmark é calculado sobre o ciclo anterior, o mesmo do restante do contexto. Ele não contém informação do ciclo que se quer prever.

In [203]:
# --- 3.3 Benchmark estadual por rede ---
# Mediana da taxa de cada rede entre os municípios da mesma UF (ciclo anterior)
base_bench = ctx_rede.merge(
    mun.loc[mun["ano"] == CICLO_ANTERIOR, ["id_municipio", "sigla_uf"]]
       .drop_duplicates(),
    on="id_municipio", how="left")

benchmark = (base_bench.groupby(["sigla_uf", "rede_nome"], observed=True)
             ["rede_taxa_ant"].median()
             .rename("uf_rede_taxa_ant").reset_index())

print(f"Benchmarks calculados: {len(benchmark)} combinações de UF e rede")
print()
print("Amostra (as cinco maiores e as cinco menores medianas):")
ordenado = benchmark.sort_values("uf_rede_taxa_ant", ascending=False)
print(pd.concat([ordenado.head(5), ordenado.tail(5)])
      .round(1).to_string(index=False))
print()

# Acoplar o benchmark ao contexto de rede
ctx_rede = base_bench.merge(benchmark, on=["sigla_uf", "rede_nome"], how="left")
ctx_rede["rede_vs_uf"] = (ctx_rede["rede_taxa_ant"]
                          - ctx_rede["uf_rede_taxa_ant"])
ctx_rede = ctx_rede.drop(columns="sigla_uf")

print("Distância da rede até o benchmark do seu estado (pontos percentuais):")
print(ctx_rede["rede_vs_uf"].describe().round(1).to_string())

Benchmarks calculados: 46 combinações de UF e rede

Amostra (as cinco maiores e as cinco menores medianas):
sigla_uf rede_nome  uf_rede_taxa_ant
      CE Municipal              93.2
      PR  Estadual              84.7
      CE  Estadual              84.2
      GO  Estadual              79.5
      ES  Estadual              79.1
      BA Municipal              36.1
      RN Municipal              36.0
      AL  Estadual              30.1
      SE Municipal              28.9
      BA  Estadual              27.5

Distância da rede até o benchmark do seu estado (pontos percentuais):
count    6597.0
mean        0.3
std        15.4
min       -66.4
25%        -9.6
50%         0.0
75%         9.7
max        58.2


### 3.4 Porte do ciclo corrente: o que já se sabe antes da prova

🎓 **Conceito, nem tudo do ciclo corrente é vazamento.** A regra continua sendo o momento em que a informação passa a existir. O **porte** da rede e do município no ciclo que se quer prever, isto é, quantos alunos há para avaliar, vem do cadastro escolar e está definido **antes** da aplicação da prova. Não depende de nenhum resultado, e por isso é informação legítima.

⚠️ **A linha fina:** alunos **avaliáveis** (matriculados, presentes e ausentes) é informação prévia; alunos **presentes** só se conhece depois da aplicação, e seria vazamento. O porte aqui conta os avaliáveis.

📌 **Por que preferir o porte corrente ao do ciclo anterior:** o porte defasado sofre de dois problemas. Envelhece (redes crescem e encolhem entre ciclos) e depende da disponibilidade dos microdados do ano anterior, o que restringe sua cobertura a 76,9% das observações. O porte corrente é calculado da própria base de alunos e cobre a totalidade. Não à toa, o porte defasado foi a variável com a menor correlação com a resposta entre todas as testadas.

Da mesma estrutura nasce uma terceira variável, que nenhuma outra expressa: a **fração do município atendida pela rede do aluno**. Ela distingue quem estuda na rede predominante do território de quem está em uma rede minoritária ali.

In [204]:
# --- 3.4 Porte do ciclo corrente (informação prévia à avaliação) ---
# Base completa do ciclo (presentes e ausentes): é o cadastro de avaliáveis
cadastro = df_alunos[df_alunos["ano"] == CICLO_ALVO]

porte_rede = (cadastro.groupby(["id_municipio", "rede_nome"], observed=True)
              .size().rename("rede_porte_atual").reset_index())
porte_mun = (cadastro.groupby("id_municipio", observed=True)
             .size().rename("mun_porte_atual").reset_index())

print(f"Avaliáveis no ciclo {CICLO_ALVO}: {len(cadastro):,} "
      f"(presentes e ausentes)")
print(f"Combinações município e rede: {len(porte_rede):,}")
print(f"Municípios: {len(porte_mun):,}")
print()
print("Porte por rede (alunos avaliáveis por município):")
print(porte_rede.groupby("rede_nome", observed=True)["rede_porte_atual"]
      .describe()[["count", "mean", "50%", "max"]].round(0).to_string())

Avaliáveis no ciclo 2024: 2,119,624 (presentes e ausentes)
Combinações município e rede: 6,543
Municípios: 5,519

Porte por rede (alunos avaliáveis por município):
            count   mean    50%      max
rede_nome                               
Estadual   1090.0  257.0   49.0  58612.0
Municipal  5452.0  337.0  114.0  49820.0
Privada       1.0   25.0   25.0     25.0


## 4. Enriquecimento com fontes externas

**Passos desta seção:** (4.1) preparar o acesso às fontes públicas e o cache no lake; (4.2) trazer o retrato da infraestrutura escolar; (4.3) o tamanho de turma da série avaliada; (4.4) o contexto econômico do município; (4.5) o contexto socioeconômico estrutural; (4.6) a exposição à violência; (4.7) o Censo Demográfico de 2022; (4.8) o corpo docente e a jornada escolar; (4.9) o nível socioeconômico das famílias atendidas pela rede; (4.10) a primeira infância.

🎓 **Por que esta seção existe.** As variáveis construídas até aqui descrevem o **desempenho anterior** do território: taxa da rede, taxa do município, meta pactuada, distância até o benchmark. São informativas, mas todas medem a mesma coisa que se quer prever, apenas deslocada no tempo. Um modelo construído só com elas aprende que **quem ia mal continua indo mal**, o que é verdadeiro e inútil: não identifica *fatores*, e não sugere ação nenhuma ao gestor, que não pode intervir sobre a taxa do ano passado.

O enunciado pede outra coisa. Ele pergunta **quais fatores mais impactam a alfabetização** e espera inteligência aplicável a políticas públicas. Isso exige variáveis de natureza diferente, que descrevam as **condições** em que o ensino acontece: a estrutura da escola, a distância que o aluno percorre, o tamanho da turma, a presença de profissionais de apoio, a riqueza e a vulnerabilidade do território. É sobre essas que uma política pode agir.

📌 **Todas as fontes respeitam a regra temporal.** Cada uma entra com informação anterior ao ciclo avaliado, e o critério de disponibilidade considera também o calendário de publicação: o Censo Escolar de 2023 é divulgado antes da avaliação de 2024, enquanto o PIB municipal tem cerca de dois anos de defasagem, e por isso entra o de 2021.

| Fonte | Referência | O que traz | Ressalva |
|---|---|---|---|
| Censo Escolar (INEP) | 2023 | infraestrutura, localização rural, transporte escolar, profissionais de apoio, salas, professores dos anos iniciais, área diferenciada | nenhuma |
| Censo Escolar, turmas | 2023 | tamanho médio da turma do 2º ano | nenhuma |
| PIB municipal (IBGE) | 2021 | PIB per capita e perfil setorial | defasagem de publicação |
| Atlas do Desenvolvimento Humano | 2010 | IDHM, renda, desigualdade, deslocamento até o trabalho | dado censitário, defasado |
| Atlas da Violência (IPEA) | 2010 | vulnerabilidade social e seus componentes | dado censitário, defasado |
| Mortalidade (SIM/DataSUS) | 2019 | óbitos por agressão, como medida de violência | defasado |
| Censo Demográfico (IBGE) | 2022 | densidade, domicílio, saneamento, alfabetização adulta, território | nenhuma |
| Indicadores Educacionais (INEP) | 2022 | formação, regularidade e esforço docente; horas-aula diárias | a publicação de 2023 está corrompida |
| Nível Socioeconômico (INEP) | 2021 | condição das famílias atendidas pela rede | edição mais recente |
| Nascidos vivos (SINASC) | 2016 e 2017 | escolaridade e idade da mãe da coorte avaliada | residência da mãe no parto |
| Censo Escolar, pré-escola (INEP) | 2022 | oferta de pré-escola no município | nenhuma |

📌 **Sobre o caminho que estes dados percorrem** (decisão D-005): as fontes externas são consultadas diretamente na origem, e o resultado agregado é materializado no data lake para reutilização. Elas não passam pelas camadas do medalhão nesta fase, porque a etapa é de prototipação: o propósito é descobrir **se** essas variáveis carregam informação útil, e industrializar a ingestão de várias fontes antes dessa resposta seria construir infraestrutura para dados que podem ser descartados na seleção de variáveis. Fica registrada a recomendação de que as fontes aprovadas pela análise de importância sejam incorporadas ao lake pelo time de engenharia de dados, com o dado bruto na camada Bronze e a agregação como transformação explícita na Silver.

⚠️ **Sobre as fontes defasadas.** IDHM, vulnerabilidade social e violência são características **estruturais** do território, que se alteram lentamente. Entram como aproximação do contexto, com a defasagem declarada aqui e nas limitações do projeto. O julgamento sobre sua utilidade não é feito por opinião: se não carregarem informação, a análise de importância dos modelos mostrará isso, e a ausência de efeito também será um achado.

In [205]:
# --- 4.1 Acesso às fontes públicas, com cache no lake ---
import pandas_gbq

def obter_fonte_externa(nome: str, consulta: str,
                        colunas: list[str] | None = None,
                        forcar: bool = False) -> pd.DataFrame:
    """Consulta uma fonte pública e guarda o resultado no lake.

    A consulta ao BigQuery é feita uma única vez: o resultado agregado é
    gravado em ml/externas/ e reutilizado nas execuções seguintes. Isso
    torna a construção da base reprodutível sem repetir o custo de leitura
    e sem depender da disponibilidade da fonte a cada execução.
    """
    caminho_blob = f"ml/externas/{nome}/{nome}.parquet"
    blob = cliente_storage.bucket(BUCKET_LAKE).blob(caminho_blob)

    if blob.exists() and not forcar:
        import io
        dados = pd.read_parquet(io.BytesIO(blob.download_as_bytes()))
        # O cache é compartilhado com o script de produção. Se o esquema
        # gravado não tiver o que esta célula espera, ele é refeito em vez
        # de quebrar lá na frente, na integração.
        faltantes = [c for c in (colunas or []) if c not in dados.columns]
        if not faltantes:
            print(f"  {nome:<22} {len(dados):>7,} linhas  (cache no lake)")
            return dados
        print(f"  {nome:<22} cache sem {faltantes}; refazendo a consulta")

    dados = pandas_gbq.read_gbq(consulta, project_id=PROJETO_GCP,
                                credentials=credenciais,
                                progress_bar_type=None)
    dados.to_parquet(f"gs://{BUCKET_LAKE}/{caminho_blob}", index=False,
                     storage_options={"token": credenciais})
    print(f"  {nome:<22} {len(dados):>7,} linhas  (consultado e gravado)")
    return dados

print("Fontes externas (cache em ml/externas/):")

Fontes externas (cache em ml/externas/):


⚠️ **Só escolas em atividade.** O cadastro do Censo Escolar mantém escolas paralisadas e extintas com o registro em branco: matrícula, salas e todos os campos de infraestrutura vêm vazios. Elas são 16,5% das escolas das redes estadual e municipal. Na primeira versão da consulta abaixo, elas entravam na conta como escolas sem água, sem energia e sem biblioteca, o que puxava para baixo as variáveis de infraestrutura em 43% das combinações de município e rede; na energia elétrica, a distorção mediana nessas combinações era de 25 pontos percentuais. A consulta passa a considerar apenas as escolas em atividade. O defeito foi encontrado na análise exploratória, ao comparar a infraestrutura das redes estadual e municipal.

In [ ]:
# --- 4.2 Censo Escolar: a infraestrutura em que o ensino acontece ---
# Códigos de rede na tabela de escolas: 2 estadual, 3 municipal.
CENSO_ESCOLAR = f"""
SELECT id_municipio, rede,
       COUNT(*) AS esc_quantidade,
       ROUND(100 * SAFE_DIVIDE(
                   SUM(IF(tipo_localizacao = '2', quantidade_matricula_fundamental_2_ano, 0)),
                   SUM(quantidade_matricula_fundamental_2_ano)), 1) AS esc_pct_rural,
               ROUND(100 * SAFE_DIVIDE(
                   SUM(quantidade_matricula_zona_residencia_rural),
                   SUM(quantidade_matricula_zona_residencia_rural)
                   + SUM(quantidade_matricula_zona_residencia_urbana)), 1) AS esc_pct_alunos_zona_rural,
       ROUND(100 * AVG(COALESCE(agua_rede_publica, 0)), 1) AS esc_pct_agua_rede,
       ROUND(100 * AVG(COALESCE(esgoto_rede_publica, 0)), 1) AS esc_pct_esgoto_rede,
       ROUND(100 * AVG(COALESCE(energia_rede_publica, 0)), 1) AS esc_pct_energia_rede,
       ROUND(100 * AVG(COALESCE(internet, 0)), 1) AS esc_pct_internet,
       -- biblioteca ou sala de leitura: muitas redes organizam o acervo em salas
       -- de leitura, e a biblioteca sozinha subestimava o acesso a livros
       ROUND(100 * AVG(COALESCE(SAFE_CAST(biblioteca_sala_leitura AS FLOAT64), 0)), 1)
           AS esc_pct_biblioteca_ou_sala_leitura,
       ROUND(100 * AVG(COALESCE(laboratorio_informatica, 0)), 1) AS esc_pct_lab_informatica,
       ROUND(100 * AVG(COALESCE(quadra_esportes, 0)), 1) AS esc_pct_quadra,
       ROUND(100 * AVG(COALESCE(alimentacao, 0)), 1) AS esc_pct_alimentacao,
       ROUND(100 * AVG(COALESCE(profissional_coordenador, 0)), 1) AS esc_pct_coordenador,
       ROUND(100 * AVG(COALESCE(profissional_psicologo, 0)), 1) AS esc_pct_psicologo,
       ROUND(100 * AVG(COALESCE(profissional_assistente_social, 0)), 1) AS esc_pct_assistente_social,
       -- terra indígena, quilombo, assentamento ou comunidade tradicional;
       -- o código 0 indica escola fora de área diferenciada
       ROUND(100 * AVG(IF(SAFE_CAST(tipo_localizacao_diferenciada AS FLOAT64) > 0, 1, 0)), 1)
           AS esc_pct_area_diferenciada,
       ROUND(SAFE_DIVIDE(SUM(quantidade_matricula_fundamental_anos_iniciais),
                         SUM(quantidade_docente_fundamental_anos_iniciais)), 1)
           AS esc_alunos_por_docente,
       SUM(quantidade_sala_utilizada) AS esc_salas,
       SUM(quantidade_matricula_utiliza_transporte_publico) AS esc_alunos_transporte,
       SUM(quantidade_matricula_educacao_basica) AS esc_matriculas
FROM `basedosdados.br_inep_censo_escolar.escola`
WHERE ano = {CICLO_ANTERIOR} AND rede IN ('2', '3')
  -- só escolas em atividade: paralisadas e extintas ficam no cadastro com
  -- matrícula, salas e infraestrutura em branco
  AND CAST(tipo_situacao_funcionamento AS STRING) = '1'
GROUP BY id_municipio, rede
"""

censo = obter_fonte_externa("censo_escolar_ativas", CENSO_ESCOLAR,
                            ["esc_pct_rural", "esc_pct_alunos_zona_rural",
                             "esc_salas", "esc_alunos_transporte",
                             "esc_matriculas", "esc_pct_biblioteca_ou_sala_leitura",
                             "esc_pct_area_diferenciada", "esc_alunos_por_docente"])

# A rede aqui vem em código; o restante da base usa o nome
censo["rede_nome"] = censo["rede"].map({"2": "Estadual", "3": "Municipal"})
censo = censo.drop(columns="rede")

print()
print(f"Municípios cobertos: {censo['id_municipio'].nunique():,}")
print()
print("Perfil médio das redes (não ponderado por porte):")
print(censo.groupby("rede_nome", observed=True)[
    ["esc_pct_rural", "esc_pct_alunos_zona_rural", "esc_pct_internet", "esc_pct_biblioteca_ou_sala_leitura",
     "esc_pct_coordenador", "esc_pct_psicologo"]].mean().round(1).to_string())

In [207]:
# --- 4.3 Tamanho da turma na série avaliada ---
# Atenção: nesta tabela a rede vem por extenso, e não em código.
# A etapa 15 corresponde ao 2º ano do ensino fundamental de nove anos,
# exatamente a série submetida à avaliação.
TURMAS = f"""
SELECT id_municipio,
       CASE rede WHEN 'estadual' THEN 'Estadual'
                 WHEN 'municipal' THEN 'Municipal' END AS rede_nome,
       ROUND(AVG(quantidade_matriculas), 1) AS turma_media_alunos
FROM `basedosdados.br_inep_censo_escolar.turma`
WHERE ano = {CICLO_ANTERIOR}
  AND rede IN ('estadual', 'municipal')
  AND etapa_ensino = '15'
GROUP BY id_municipio, rede_nome
"""

turmas = obter_fonte_externa("turmas_2ano", TURMAS, ["turma_media_alunos"])
print()
print("Tamanho médio da turma do 2º ano, por rede:")
print(turmas.groupby("rede_nome", observed=True)["turma_media_alunos"]
      .describe()[["count", "mean", "50%", "max"]].round(1).to_string())

  turmas_2ano              6,847 linhas  (cache no lake)

Tamanho médio da turma do 2º ano, por rede:
            count  mean   50%   max
rede_nome                          
Estadual   1326.0  18.4  19.0  50.0
Municipal  5521.0  19.0  19.1  34.0


In [208]:
# --- 4.4 Contexto econômico do município ---
# O PIB municipal é publicado com cerca de dois anos de defasagem: o ano de
# referência usado é o mais recente disponível antes da avaliação.
ANO_PIB = CICLO_ALVO - 3
ECONOMIA = f"""
SELECT p.id_municipio,
       ROUND(p.pib / NULLIF(pop.populacao, 0), 0) AS mun_pib_per_capita,
       ROUND(100 * p.va_agropecuaria / NULLIF(p.va, 0), 1) AS mun_pct_agropecuaria,
       ROUND(100 * p.va_servicos / NULLIF(p.va, 0), 1) AS mun_pct_servicos,
       pop.populacao AS mun_populacao
FROM `basedosdados.br_ibge_pib.municipio` p
JOIN `basedosdados.br_ibge_populacao.municipio` pop
  ON p.id_municipio = pop.id_municipio AND p.ano = pop.ano
WHERE p.ano = {ANO_PIB}
"""

economia = obter_fonte_externa("economia_municipal", ECONOMIA,
                               ["mun_pib_per_capita", "mun_populacao"])
print()
print(f"Referência: {ANO_PIB} (o PIB municipal é divulgado com defasagem)")
print(economia[["mun_pib_per_capita", "mun_pct_agropecuaria",
                "mun_populacao"]].describe(
    percentiles=[.25, .5, .75]).round(0).to_string())

  economia_municipal       5,570 linhas  (cache no lake)

Referência: 2021 (o PIB municipal é divulgado com defasagem)
       mun_pib_per_capita  mun_pct_agropecuaria  mun_populacao
count              5570.0                5570.0         5570.0
mean              33873.0                  24.0        38298.0
std               41909.0                  19.0       224288.0
min                5406.0                 -42.0          771.0
25%               12832.0                   9.0         5454.0
50%               23373.0                  19.0        11732.0
75%               40810.0                  36.0        25765.0
max              920828.0                  94.0     12396372.0


In [ ]:
# --- 4.5 Contexto socioeconômico estrutural ---
# Fontes censitárias, com referência em 2010. Entram como aproximação de
# características que mudam devagar, com a defasagem declarada.
DESENVOLVIMENTO = """
SELECT a.id_municipio,
       a.idhm AS mun_idhm,
       a.idhm_e AS mun_idhm_educacao,
       a.idhm_r AS mun_idhm_renda,
       a.renda_pc AS mun_renda_per_capita,
       a.indice_gini AS mun_gini,
       a.expectativa_anos_estudo AS mun_expectativa_estudo,
       a.taxa_vulner_desloc_1_hora AS mun_pct_vulner_deslocamento_1h,
       v.ivs AS mun_ivs,
       v.ivs_infraestrutura_urbana AS mun_ivs_infraestrutura,
       v.ivs_capital_humano AS mun_ivs_capital_humano
FROM `basedosdados.mundo_onu_adh.municipio` a
LEFT JOIN (
    SELECT id_municipio, AVG(ivs) AS ivs,
           AVG(ivs_infraestrutura_urbana) AS ivs_infraestrutura_urbana,
           AVG(ivs_capital_humano) AS ivs_capital_humano
    FROM `basedosdados.br_ipea_avs.municipio`
    WHERE ano = 2010 GROUP BY id_municipio
) v ON a.id_municipio = v.id_municipio
WHERE a.ano = 2010
"""

desenvolvimento = obter_fonte_externa("desenvolvimento_humano", DESENVOLVIMENTO,
                                      ["mun_idhm", "mun_ivs",
                                       "mun_pct_vulner_deslocamento_1h"])
print()
print("Indicadores estruturais (referência 2010):")
print(desenvolvimento[["mun_idhm", "mun_renda_per_capita", "mun_gini",
                       "mun_pct_vulner_deslocamento_1h", "mun_ivs"]]
      .describe(percentiles=[.25, .5, .75]).round(2).to_string())

In [210]:
# --- 4.6 Exposição à violência ---
# Óbitos por agressão (CID-10 X85 a Y09) no Sistema de Informação sobre
# Mortalidade, convertidos em taxa por cem mil habitantes.
ANO_VIOLENCIA = 2019
VIOLENCIA = f"""
SELECT id_municipio, SUM(numero_obitos) AS obitos_agressao
FROM `basedosdados.br_ms_sim.municipio_causa`
WHERE ano = {ANO_VIOLENCIA}
  AND REGEXP_CONTAINS(causa_basica, r'^(X8[5-9]|X9[0-9]|Y0[0-9])')
GROUP BY id_municipio
"""

violencia = obter_fonte_externa("violencia_municipal", VIOLENCIA,
                                ["obitos_agressao"])

# Municípios sem registro não são desconhecidos: não houve óbito por agressão
violencia = economia[["id_municipio", "mun_populacao"]].merge(
    violencia, on="id_municipio", how="left")
violencia["obitos_agressao"] = violencia["obitos_agressao"].fillna(0)
violencia["mun_taxa_homicidio"] = (100_000 * violencia["obitos_agressao"]
                                   / violencia["mun_populacao"]).round(1)
violencia = violencia[["id_municipio", "mun_taxa_homicidio"]]

print()
print(f"Taxa de homicídio por cem mil habitantes ({ANO_VIOLENCIA}):")
print(violencia["mun_taxa_homicidio"].describe(
    percentiles=[.25, .5, .75, .95]).round(1).to_string())
print()
print(f"Municípios sem óbito por agressão registrado: "
      f"{(violencia['mun_taxa_homicidio'] == 0).sum():,} "
      f"({(violencia['mun_taxa_homicidio'] == 0).mean():.1%})")

  violencia_municipal      3,887 linhas  (cache no lake)

Taxa de homicídio por cem mil habitantes (2019):
count    5570.0
mean       18.1
std        20.4
min         0.0
25%         0.0
50%        13.2
75%        27.4
95%        55.5
max       193.9

Municípios sem óbito por agressão registrado: 1,683 (30.2%)


### 4.7 Censo Demográfico de 2022

As variáveis socioeconômicas trazidas até aqui vêm do Atlas do Desenvolvimento Humano, cuja referência é o Censo de **2010**. A criança avaliada em 2024 nasceu por volta de 2017, e a defasagem de quatorze anos é a limitação mais séria da base.

O Censo de 2022 permite reduzi-la. Ele não traz renda no grão municipal, que continua vindo de 2010, mas traz alfabetização adulta, saneamento domiciliar, composição do domicílio e, sobretudo, **densidade demográfica**.

🎓 **Por que a densidade importa aqui:** a hipótese H1 propõe que o custo do deslocamento até a escola prejudica a aprendizagem, e que esse custo decorre do menor adensamento populacional. Até agora a hipótese era testada apenas por aproximações: percentual de escolas rurais e dependência de transporte. A densidade é a **medida direta** do conceito que a hipótese invoca.

⚠️ **Duas conferências feitas antes de escrever esta consulta.** A primeira: a chave `id_municipio` do Censo tem o mesmo formato de sete dígitos da base analítica, e cobre 100% dos municípios avaliados. A segunda: as categorias de esgotamento sanitário e de abastecimento de água **não são excludentes**. A soma delas chega ao dobro da população do município, porque há categorias que contêm outras. Somá-las produziria percentuais acima de 100%. Por isso cada indicador usa uma única categoria como numerador e a população do município como denominador.

📌 **O Censo de 2022 também passa a ser a fonte do território.** Até aqui, a UF e a região vinham da tabela municipal do ciclo anterior, que só cobre os municípios avaliados naquele ano. A análise exploratória mostrou que isso deixava 356 municípios, com cerca de 10% dos alunos, sem UF, e impedia a imputação por mediana estadual justamente nos casos que precisavam dela. O Censo cobre todos os municípios; a região é derivada do primeiro dígito do código IBGE.

In [ ]:
# --- 4.7 Censo Demográfico de 2022 ---
# As categorias de esgoto e água são aninhadas na fonte: a soma delas chega
# ao dobro da população. Cada indicador usa uma categoria específica como
# numerador e a população do município como denominador.
CENSO_2022 = """
WITH base AS (
  SELECT id_municipio, sigla_uf, populacao, domicilios, area,
         ROUND(100 * taxa_alfabetizacao, 1) AS mun_alfabetizacao_adulta,
         idade_mediana AS mun_idade_mediana
  FROM `basedosdados.br_ibge_censo_2022.municipio`
),
esgoto AS (
  SELECT id_municipio, SUM(populacao) AS com_esgoto
  FROM `basedosdados.br_ibge_censo_2022.caracteristica_domicilio_grupo_idade_raca_esgotamento_sanitario`
  WHERE tipo_esgotamento_sanitario = 'Rede geral, rede pluvial ou fossa ligada à rede'
  GROUP BY id_municipio
),
agua AS (
  SELECT id_municipio, SUM(populacao) AS com_agua
  FROM `basedosdados.br_ibge_censo_2022.caracteristica_domicilio_grupo_idade_raca_ligacao_abastecimento_agua`
  WHERE tipo_ligacao_rede_geral = 'Possui ligação à rede geral e a utiliza como forma principal'
  GROUP BY id_municipio
)
SELECT b.id_municipio,
       b.sigla_uf,
       ROUND(b.populacao / NULLIF(b.area, 0), 1)       AS mun_densidade_demografica,
       ROUND(b.populacao / NULLIF(b.domicilios, 0), 2) AS mun_pessoas_por_domicilio,
       b.mun_alfabetizacao_adulta,
       b.mun_idade_mediana,
       ROUND(100 * e.com_esgoto / NULLIF(b.populacao, 0), 1) AS mun_pct_esgoto_adequado,
       ROUND(100 * a.com_agua  / NULLIF(b.populacao, 0), 1)  AS mun_pct_agua_rede_geral
FROM base b
LEFT JOIN esgoto e USING (id_municipio)
LEFT JOIN agua   a USING (id_municipio)
"""

censo_2022 = obter_fonte_externa("censo_demografico_2022", CENSO_2022,
                                 ["sigla_uf", "mun_densidade_demografica",
                                  "mun_alfabetizacao_adulta"])

# A região decorre do primeiro dígito do código IBGE do município
REGIOES = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul",
           "5": "Centro-Oeste"}
censo_2022["nome_regiao"] = censo_2022["id_municipio"].str[0].map(REGIOES)

print()
print("Referência: 2022, contra 2010 das variáveis do Atlas")
print(censo_2022.drop(columns=["id_municipio", "sigla_uf",
                                "nome_regiao"]).describe(
    percentiles=[.25, .5, .75]).round(1).to_string())
print()
print(f"Municípios cobertos: {censo_2022['id_municipio'].nunique():,}")
print(f"Municípios sem UF: {censo_2022['sigla_uf'].isna().sum()}   "
      f"sem região: {censo_2022['nome_regiao'].isna().sum()}")

### 4.8 Corpo docente e jornada escolar

As variáveis de escola trazidas até aqui descrevem o prédio e os profissionais de apoio, mas nenhuma descreve quem alfabetiza. O INEP publica, por município e rede, indicadores calculados a partir do cadastro de professores do Censo Escolar: a formação do professor diante da disciplina que leciona, a permanência dele na mesma escola ao longo dos anos e a sua carga de trabalho. Publica também a média de horas-aula diárias, que mede o tempo que a criança passa na escola. Essas variáveis respondem às hipóteses H10 (corpo docente) e H11 (tempo na escola).

⚠️ **Referência 2022, e não 2023.** A tabela publicada para 2023 e 2024 está corrompida na origem: os cinco grupos de formação docente, que deveriam somar 100%, somam entre 287% e 500%, e as horas-aula aparecem zeradas. Os anos de 2019 a 2022 estão consistentes, com somas exatas de 100%. Uso 2022, o mais recente íntegro, e a célula confere a soma dos grupos antes de aceitar o dado.

📌 **Nem toda rede tem valor.** Os indicadores de formação, esforço e horas-aula são próprios dos anos iniciais. Onde a rede estadual não oferece essa etapa, eles vêm vazios. A cobertura entre os alunos da base é conferida na integração (5.1).

In [ ]:
# --- 4.8 Corpo docente e jornada escolar ---
# Indicadores do INEP por município e rede. Nesta tabela a rede vem por
# extenso, em minúsculas, e a localização 'total' reúne escolas urbanas e
# rurais. Referência 2022: a publicação de 2023 e 2024 está corrompida.
ANO_INDICADORES = 2022
INDICADORES_DOCENTES = f"""
SELECT id_municipio,
       INITCAP(CAST(rede AS STRING)) AS rede_nome,
       SAFE_CAST(afd_ef_anos_iniciais_grupo_1 AS FLOAT64) AS esc_pct_docentes_formacao_adequada,
       SAFE_CAST(ird_baixa_regularidade AS FLOAT64)
         + SAFE_CAST(ird_media_baixa AS FLOAT64) AS esc_pct_docentes_baixa_regularidade,
       SAFE_CAST(ied_ef_anos_iniciais_nivel_5 AS FLOAT64)
         + SAFE_CAST(ied_ef_anos_iniciais_nivel_6 AS FLOAT64) AS esc_pct_docentes_alto_esforco,
       SAFE_CAST(had_ef_anos_iniciais AS FLOAT64) AS esc_horas_aula_diarias,
       -- conferência de integridade: os cinco grupos de formação somam 100
       SAFE_CAST(afd_ef_anos_iniciais_grupo_1 AS FLOAT64)
         + SAFE_CAST(afd_ef_anos_iniciais_grupo_2 AS FLOAT64)
         + SAFE_CAST(afd_ef_anos_iniciais_grupo_3 AS FLOAT64)
         + SAFE_CAST(afd_ef_anos_iniciais_grupo_4 AS FLOAT64)
         + SAFE_CAST(afd_ef_anos_iniciais_grupo_5 AS FLOAT64) AS soma_grupos_formacao
FROM `basedosdados.br_inep_indicadores_educacionais.municipio`
WHERE ano = {ANO_INDICADORES}
  AND LOWER(CAST(localizacao AS STRING)) = 'total'
  AND LOWER(CAST(rede AS STRING)) IN ('estadual', 'municipal')
"""

docentes = obter_fonte_externa(
    "indicadores_docentes", INDICADORES_DOCENTES,
    ["rede_nome", "esc_pct_docentes_formacao_adequada",
     "esc_pct_docentes_baixa_regularidade", "esc_pct_docentes_alto_esforco",
     "esc_horas_aula_diarias", "soma_grupos_formacao"])

# Travas: uma linha por município e rede, e grupos de formação somando 100
if docentes.duplicated(["id_municipio", "rede_nome"]).any():
    raise RuntimeError("Indicadores docentes com mais de uma linha por município e rede")
fora_de_100 = (docentes["soma_grupos_formacao"].dropna() - 100).abs().gt(1).sum()
if fora_de_100:
    raise RuntimeError(f"{fora_de_100} redes com grupos de formação que não somam 100: "
                       "a publicação pode estar corrompida, como a de 2023")
docentes = docentes.drop(columns="soma_grupos_formacao")

print()
print(f"Referência: {ANO_INDICADORES}. Grupos de formação somando 100 em todas as redes.")
print("Mediana por rede:")
print(docentes.groupby("rede_nome")[[
    "esc_pct_docentes_formacao_adequada", "esc_pct_docentes_baixa_regularidade",
    "esc_pct_docentes_alto_esforco", "esc_horas_aula_diarias"]]
    .median().round(1).T.to_string())

### 4.9 Nível socioeconômico das famílias atendidas pela rede

As variáveis socioeconômicas das seções 4.4, 4.5 e 4.7 descrevem o município inteiro e são as mesmas para o aluno da rede estadual e para o da municipal. O Indicador de Nível Socioeconômico (INSE) do INEP descreve as famílias que cada rede atende: é calculado com o questionário respondido pelos alunos no SAEB, que pergunta sobre a escolaridade dos pais, a renda e os bens do domicílio. É a única medida socioeconômica desta base no grão da rede e responde à hipótese H9.

📌 A edição mais recente é a de 2021. Na fonte, 148 combinações de município e rede têm INSE igual a zero, valor fora da escala (o menor valor válido é 3,35). Elas são tratadas como ausentes.

In [ ]:
# --- 4.9 Nível socioeconômico das famílias atendidas pela rede ---
# Nesta tabela a rede vem em código (2 estadual, 3 municipal) e a
# localização 0 é o total. INSE igual a zero está fora da escala do
# indicador e é tratado como ausente.
ANO_INSE = 2021
INSE = f"""
SELECT id_municipio,
       CASE CAST(rede AS STRING) WHEN '2' THEN 'Estadual'
                                 WHEN '3' THEN 'Municipal' END AS rede_nome,
       NULLIF(SAFE_CAST(inse AS FLOAT64), 0) AS esc_inse_medio
FROM `basedosdados.br_inep_indicador_nivel_socioeconomico.municipio`
WHERE ano = {ANO_INSE}
  AND CAST(tipo_localizacao AS STRING) = '0'
  AND CAST(rede AS STRING) IN ('2', '3')
"""

inse = obter_fonte_externa("inse_rede", INSE, ["rede_nome", "esc_inse_medio"])
if inse.duplicated(["id_municipio", "rede_nome"]).any():
    raise RuntimeError("INSE com mais de uma linha por município e rede")

print()
print(f"Referência: {ANO_INSE}. Ausentes (zero na fonte): {inse['esc_inse_medio'].isna().sum()}")
print(inse.groupby("rede_nome")["esc_inse_medio"]
      .describe()[["count", "min", "25%", "50%", "75%", "max"]].round(2).to_string())

### 4.10 Primeira infância: a mãe no nascimento e a pré-escola

A criança avaliada no 2º ano em 2024 nasceu, em geral, em 2016 ou 2017. Duas fontes descrevem as condições em que essa geração chegou à escola.

- O **Sistema de Informações sobre Nascidos Vivos (SINASC)**, do Ministério da Saúde, registra cada nascimento com a idade e a escolaridade da mãe. Agregando os nascidos de 2016 e 2017 pelo município de residência da mãe, obtenho o percentual de mães com ao menos 8 anos de estudo, a medida mais direta de capital cultural disponível (H6), e o percentual de mães adolescentes (H12).
- A **pré-escola** (4 e 5 anos de idade) é a etapa em que a criança tem o primeiro contato sistemático com letras, sons e livros. O Censo Escolar permite medir a oferta de pré-escola do município diante dos anos iniciais (H12). A referência é 2022, o último ano em que a coorte avaliada estava na pré-escola.

📌 **Por que a oferta de pré-escola soma todas as redes.** A criança pode ter feito a pré-escola numa rede diferente daquela em que cursa o 2º ano, inclusive na privada. A variável descreve o município, e não a rede.

⚠️ **Limite:** o SINASC registra o município de residência da mãe no parto. Crianças que mudaram de município até os 7 anos continuam atribuídas ao município de origem.

In [ ]:
# --- 4.10 Primeira infância: a mãe no nascimento e a pré-escola ---
# A criança avaliada no 2º ano em 2024 nasceu, em geral, em 2016 ou 2017.
# Escolaridade da mãe no SINASC, em anos de estudo: 1 nenhum, 2 de 1 a 3,
# 3 de 4 a 7, 4 de 8 a 11, 5 doze ou mais, 9 ignorado (fora do denominador).
ANOS_NASCIMENTO = (CICLO_ALVO - 8, CICLO_ALVO - 7)
NASCIMENTOS = f"""
SELECT CAST(id_municipio_residencia AS STRING) AS id_municipio,
       ROUND(100 * SAFE_DIVIDE(
           COUNTIF(CAST(escolaridade_mae AS STRING) IN ('4', '5')),
           COUNTIF(CAST(escolaridade_mae AS STRING) IN ('1', '2', '3', '4', '5'))), 1)
           AS mun_pct_maes_8_anos_estudo,
       ROUND(100 * SAFE_DIVIDE(
           COUNTIF(SAFE_CAST(idade_mae AS INT64) BETWEEN 10 AND 19),
           COUNTIF(SAFE_CAST(idade_mae AS INT64) BETWEEN 10 AND 60)), 1)
           AS mun_pct_maes_adolescentes
FROM `basedosdados.br_ms_sinasc.microdados`
WHERE ano IN {ANOS_NASCIMENTO} AND id_municipio_residencia IS NOT NULL
GROUP BY 1
"""

# Pré-escola por ano de idade (a etapa dura 2 anos) sobre anos iniciais por
# série (5 anos), somando todas as redes do município, escolas em atividade.
# Referência: o último ano em que a coorte avaliada estava na pré-escola.
ANO_PRE_ESCOLA = CICLO_ALVO - 2
PRE_ESCOLA = f"""
SELECT id_municipio,
       ROUND(SAFE_DIVIDE(SUM(quantidade_matricula_infantil_pre_escola) / 2,
                         SUM(quantidade_matricula_fundamental_anos_iniciais) / 5), 2)
           AS mun_indice_pre_escola
FROM `basedosdados.br_inep_censo_escolar.escola`
WHERE ano = {ANO_PRE_ESCOLA}
  AND CAST(tipo_situacao_funcionamento AS STRING) = '1'
GROUP BY id_municipio
"""

nascimentos = obter_fonte_externa(
    "nascimentos_coorte", NASCIMENTOS,
    ["mun_pct_maes_8_anos_estudo", "mun_pct_maes_adolescentes"])
pre_escola = obter_fonte_externa("pre_escola", PRE_ESCOLA, ["mun_indice_pre_escola"])

# Trava: a chave do SINASC precisa ter o formato IBGE de sete dígitos
fora_do_formato = (~nascimentos["id_municipio"].astype(str).str.fullmatch(r"\d{7}")).sum()
if fora_do_formato:
    raise RuntimeError(f"{fora_do_formato} códigos de município do SINASC fora do formato IBGE")
primeira_infancia = nascimentos.merge(pre_escola, on="id_municipio", how="outer")

print()
print(f"Nascidos em {ANOS_NASCIMENTO[0]} e {ANOS_NASCIMENTO[1]}; pré-escola em {ANO_PRE_ESCOLA}")
print(f"Municípios cobertos: {primeira_infancia['id_municipio'].nunique():,}")
print(primeira_infancia.drop(columns="id_municipio")
      .describe(percentiles=[.05, .25, .5, .75, .95]).round(2).to_string())

## 5. Integração e auditoria contra vazamento

**Passos desta seção:** (5.1) integrar cada aluno ao contexto da sua rede e do seu município, conferindo a cobertura; (5.2) submeter a tabela a uma auditoria explícita contra vazamento.

🎓 **Conceito, vazamento de dados** (*data leakage*): ocorre quando uma variável explicativa carrega informação que, no momento real da predição, ainda não existiria. O modelo aprende um atalho, exibe métricas excelentes na avaliação e falha no uso real. O enunciado exige o tratamento desse problema, e aqui ele tem **três fontes distintas**, cada uma com sua defesa:

| Fonte de vazamento | Como se manifestaria aqui | Defesa adotada |
|---|---|---|
| **Temporal** | usar o desempenho do território no próprio ciclo do aluno | contexto sempre defasado (seção 3) |
| **Da variável resposta** | usar a proficiência do aluno, da qual a resposta é derivada | exclusão explícita, auditada na célula 5.2 |
| **Entre partições** | alunos do mesmo município em treino e em teste | separação por município (seção 6) |

⚠️ **A armadilha mais perigosa é a segunda.** A variável resposta é derivada da proficiência (alfabetizado quando a nota atinge 743 pontos). Se a proficiência entrar como variável explicativa, o modelo atinge acurácia praticamente perfeita sem aprender nada: é o gabarito dentro da prova. Por isso a auditoria mantém uma lista de colunas proibidas e verifica, em código, que nenhuma delas alcançou a tabela final.

In [ ]:
# --- 5.1 Integrar aluno, rede, município e contexto externo ---
alunos_ciclo = df_pop[df_pop["ano"] == CICLO_ALVO].copy()
print(f"Alunos presentes em {CICLO_ALVO}: {len(alunos_ciclo):,}")
antes = len(alunos_ciclo)

# Cada fonte entra pela chave do seu próprio grão
FONTES = [
    ("contexto da rede", ctx_rede, ["id_municipio", "rede_nome"]),
    ("contexto do município", ctx_mun, ["id_municipio"]),
    ("porte da rede", porte_rede, ["id_municipio", "rede_nome"]),
    ("porte do município", porte_mun, ["id_municipio"]),
    ("censo escolar", censo, ["id_municipio", "rede_nome"]),
    ("turmas do 2º ano", turmas, ["id_municipio", "rede_nome"]),
    ("economia municipal", economia, ["id_municipio"]),
    ("desenvolvimento humano", desenvolvimento, ["id_municipio"]),
    ("censo demográfico 2022", censo_2022, ["id_municipio"]),
    ("violência", violencia, ["id_municipio"]),
    ("corpo docente e jornada", docentes, ["id_municipio", "rede_nome"]),
    ("nível socioeconômico da rede", inse, ["id_municipio", "rede_nome"]),
    ("primeira infância", primeira_infancia, ["id_municipio"]),
]

abt = alunos_ciclo
for rotulo, tabela, chaves in FONTES:
    abt = abt.merge(tabela, on=chaves, how="left")
    if len(abt) != antes:
        raise RuntimeError(f"O join com '{rotulo}' alterou a contagem: "
                           f"{antes:,} -> {len(abt):,}")

print(f"Linhas após {len(FONTES)} integrações: {len(abt):,}  (esperado: igual)")
print()

# Derivadas que só existem depois de reunir as fontes
abt["rede_vs_municipio"] = abt["rede_taxa_ant"] - abt["mun_taxa_ant"]
abt["rede_peso_no_municipio"] = (abt["rede_porte_atual"]
                                 / abt["mun_porte_atual"])
# Densidade física da rede. Numerador e denominador vêm do mesmo Censo,
# no mesmo grão e no mesmo ano: usar o porte do 2º ano no denominador
# misturaria a matrícula de todas as séries com a de uma só.
abt["esc_alunos_por_sala"] = (abt["esc_matriculas"]
                              / abt["esc_salas"].replace(0, pd.NA))
# Dependência de transporte escolar, aproximação da distância até a escola
abt["esc_pct_transporte"] = (100 * abt["esc_alunos_transporte"]
                             / abt["esc_matriculas"].replace(0, pd.NA))
abt = abt.drop(columns=["esc_salas", "esc_alunos_transporte",
                        "esc_matriculas"])

print("Cobertura de cada bloco de variáveis:")
BLOCOS = {
    "desempenho anterior da rede": "rede_taxa_ant",
    "desempenho anterior do município": "mun_taxa_ant",
    "infraestrutura escolar": "esc_pct_internet",
    "tamanho de turma": "turma_media_alunos",
    "economia municipal": "mun_pib_per_capita",
    "desenvolvimento humano": "mun_idhm",
    "violência": "mun_taxa_homicidio",
    "formação docente": "esc_pct_docentes_formacao_adequada",
    "horas-aula diárias": "esc_horas_aula_diarias",
    "nível socioeconômico da rede": "esc_inse_medio",
    "nascimentos da coorte": "mun_pct_maes_adolescentes",
    "pré-escola": "mun_indice_pre_escola",
}
for rotulo, coluna in BLOCOS.items():
    cobertura = abt[coluna].notna()
    print(f"  {rotulo:<34} {cobertura.sum():>9,} alunos  "
          f"({cobertura.mean():.1%})")

In [ ]:
# --- 5.2 Auditoria contra vazamento ---
PROIBIDAS = {
    "proficiencia": "a variável resposta é derivada dela (nota >= 743)",
    "alfabetizado": "é a própria variável resposta, em outro formato",
    "presente": "constante na população modelada (todos são presentes)",
    "peso_aluno": "metadado amostral; reservado à ponderação, não é atributo",
}

CHAVES = ["ano", "id_municipio"]
RESPOSTA = "alvo"

# As variáveis explicativas, organizadas pela natureza do que descrevem
FAMILIAS = {
    "aluno": ["rede_nome"],
    "desempenho anterior": [
        "rede_taxa_ant", "rede_media_portugues_ant", "uf_rede_taxa_ant",
        "rede_vs_uf", "rede_vs_municipio", "mun_taxa_ant",
        "mun_participacao_ant", "mun_taxa_ajustada_ant", "mun_alunos_ant",
        "mun_meta_ciclo", "mun_gap_meta"],
    "território": ["sigla_uf", "nome_regiao"],
    "porte e oferta": [
        "rede_porte_atual", "mun_porte_atual", "rede_peso_no_municipio",
        "esc_quantidade", "esc_alunos_por_sala", "turma_media_alunos"],
    "infraestrutura escolar": [
        "esc_pct_rural", "esc_pct_alunos_zona_rural", "esc_pct_agua_rede", "esc_pct_esgoto_rede",
        "esc_pct_energia_rede", "esc_pct_internet", "esc_pct_biblioteca_ou_sala_leitura",
        "esc_pct_lab_informatica", "esc_pct_quadra", "esc_pct_alimentacao",
        "esc_pct_transporte", "esc_pct_area_diferenciada"],
    "profissionais de apoio": [
        "esc_pct_coordenador", "esc_pct_psicologo",
        "esc_pct_assistente_social"],
    "corpo docente e jornada": [
        "esc_pct_docentes_formacao_adequada", "esc_pct_docentes_baixa_regularidade",
        "esc_pct_docentes_alto_esforco", "esc_alunos_por_docente",
        "esc_horas_aula_diarias"],
    "contexto socioeconômico": [
        "mun_pib_per_capita", "mun_pct_agropecuaria", "mun_pct_servicos",
        "mun_populacao", "mun_idhm", "mun_idhm_educacao", "mun_idhm_renda",
        "mun_renda_per_capita", "mun_gini", "mun_expectativa_estudo",
        "mun_ivs", "mun_ivs_infraestrutura", "mun_ivs_capital_humano",
        "mun_taxa_homicidio", "mun_densidade_demografica",
        "mun_pessoas_por_domicilio", "mun_alfabetizacao_adulta",
        "mun_idade_mediana", "mun_pct_esgoto_adequado",
        "mun_pct_agua_rede_geral", "esc_inse_medio",
        "mun_pct_vulner_deslocamento_1h"],
    "primeira infância": [
        "mun_pct_maes_8_anos_estudo", "mun_pct_maes_adolescentes",
        "mun_indice_pre_escola"],
}
FEATURES = [v for grupo in FAMILIAS.values() for v in grupo]

print("Auditoria contra vazamento")
print("=" * 62)

ausentes = [v for v in FEATURES if v not in abt.columns]
if ausentes:
    raise RuntimeError(f"Variáveis declaradas mas ausentes na tabela: {ausentes}")

invasoras = [c for c in FEATURES if c in PROIBIDAS]
print(f"1. Colunas proibidas entre as variáveis: {invasoras or 'nenhuma'}")
for coluna, motivo in PROIBIDAS.items():
    print(f"     {coluna:<14} fora do modelo, {motivo}")

# Toda variável do ciclo corrente precisa de justificativa prévia à prova
EX_ANTE_DO_CICLO = {
    "rede_porte_atual", "mun_porte_atual", "rede_peso_no_municipio",
    "mun_meta_ciclo", "mun_gap_meta"}
DERIVADAS = {"rede_vs_municipio", "rede_vs_uf", "esc_alunos_por_sala",
             "esc_pct_transporte"}
CATEGORICAS_OK = {"rede_nome", "sigla_uf", "nome_regiao"}
suspeitas = [c for c in FEATURES
             if not c.endswith("_ant") and not c.startswith(("esc_", "mun_", "turma_", "uf_"))
             and c not in EX_ANTE_DO_CICLO | DERIVADAS | CATEGORICAS_OK]
print(f"2. Variáveis sem justificativa temporal: {suspeitas or 'nenhuma'}")
print("     admitidas do ciclo corrente: meta pactuada e porte do cadastro")
print(f"     fontes externas: referências anteriores ao ciclo "
      f"({CICLO_ANTERIOR} para o censo escolar, {ANO_PIB} para a economia, "
      f"2010 e {ANO_VIOLENCIA} para as estruturais, {ANO_INDICADORES} para os "
      f"indicadores docentes, {ANO_INSE} para o nível socioeconômico, "
      f"nascidos em {ANOS_NASCIMENTO[0]} e {ANOS_NASCIMENTO[1]})")

numericas = [c for c in FEATURES if pd.api.types.is_numeric_dtype(abt[c])]
correl = abt[numericas + [RESPOSTA]].corr()[RESPOSTA].drop(RESPOSTA)
print(f"3. Correlação com a resposta: {len(numericas)} variáveis numéricas")
print("   as dez mais associadas:")
print(correl.reindex(correl.abs().sort_values(ascending=False).index)
      .head(10).round(3).to_string())
suspeita_alta = correl[correl.abs() > 0.9]
print(f"   correlações acima de 0,9: {list(suspeita_alta.index) or 'nenhuma'}")

print("=" * 62)
if invasoras or suspeitas or len(suspeita_alta):
    raise RuntimeError("Auditoria reprovada: revise as variáveis acima.")
print(f"Auditoria aprovada. {len(FEATURES)} variáveis explicativas em "
      f"{len(FAMILIAS)} famílias, {len(abt):,} observações.")
for familia, variaveis in FAMILIAS.items():
    print(f"     {familia:<26} {len(variaveis):>2} variáveis")

### 5.3 O que a integração revelou

O contexto no grão da rede cobre mais alunos do que o agregado municipal cobria, porque a maior parte da população estuda na rede municipal, que tem presença em praticamente todos os municípios avaliados. As lacunas remanescentes concentram-se em dois grupos: alunos da rede estadual em municípios cuja rede estadual não foi medida no ciclo anterior, e alunos da rede privada, residual nesta base e sem contexto correspondente.

As variáveis derivadas dos microdados no nível do município (participação, taxa ajustada e total de alunos presentes) permanecem com preenchimento menor, por causa dos municípios que constam no indicador consolidado sem microdados públicos, um ponto cego identificado na fase anterior. Como vários deles são populosos, o efeito sobre a contagem de alunos é maior do que sobre a de municípios.

O tratamento desses ausentes é assunto da etapa de pré-processamento, com duas alternativas a comparar: imputação por medida de tendência central dentro do próprio estado, ou descarte das variáveis que a análise exploratória mostrar pouco informativas. A decisão será registrada no diário com a evidência que a sustentar.

## 6. Partição dos dados: por município, não por sorteio

**Passos desta seção:** (6.1) separar treino, validação e teste por município; (6.2) conferir que a partição é honesta e equilibrada.

🎓 **Conceito, por que não o `train_test_split`:** a função mais conhecida do scikit-learn sorteia **linhas** ao acaso, e é a escolha correta quando cada observação é independente das demais. Não é o caso aqui: os alunos estão **aninhados em municípios**, e nove das dez variáveis explicativas são municipais, idênticas para todos os alunos de um mesmo território.

O efeito fica claro com um exemplo desta base. São Paulo contribui com cerca de cem mil alunos, todos com os mesmos valores de contexto. Num sorteio por linha, parte deles ficaria no treino e parte no teste; o modelo aprenderia a associação entre aquele conjunto específico de valores e o resultado, e voltaria a encontrá-lo na avaliação. A métrica subiria sem que houvesse generalização: o modelo teria memorizado o município, não aprendido o fenômeno. O efeito tem nome, **vazamento por agrupamento**, e é a terceira fonte listada na seção 5.

A resposta do próprio scikit-learn para dados agrupados é o `GroupShuffleSplit`, do mesmo módulo `model_selection`: em vez de sortear linhas, ele sorteia **grupos**, mantendo todas as observações de um município do mesmo lado da fronteira. O critério de escolha entre as duas funções é a estrutura do dado, não o costume:

| Estrutura das observações | Função adequada |
|---|---|
| Independentes (uma linha, uma entidade) | `train_test_split` |
| Aninhadas em grupos (alunos em municípios, consultas em pacientes) | `GroupShuffleSplit`, `GroupKFold` |

A partição por município resolve isso e, mais do que resolver, **mede o que interessa**: o conjunto de teste passa a ser formado por municípios que o modelo nunca viu, que é a situação real de uso, quando se pretende antecipar o risco de territórios ainda não avaliados no ciclo.

📌 **Proporção adotada:** 60% dos municípios para treino, 20% para validação e 20% para teste. A validação serve à escolha de modelos e ao ajuste de hiperparâmetros; o teste permanece intocado até a avaliação final, para preservar a honestidade da estimativa de desempenho.

In [214]:
# --- 6.1 Separar treino, validação e teste por município ---
from sklearn.model_selection import GroupShuffleSplit

SEMENTE = 42  # replicabilidade: mesma semente, mesma partição

municipios = abt["id_municipio"]

# Primeira divisão: 60% treino, 40% para dividir entre validação e teste
divisor = GroupShuffleSplit(n_splits=1, train_size=0.6, random_state=SEMENTE)
idx_treino, idx_resto = next(divisor.split(abt, groups=municipios))

# Segunda divisão: metade do resto para validação, metade para teste
resto = abt.iloc[idx_resto]
divisor2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEMENTE)
idx_val, idx_teste = next(divisor2.split(resto, groups=resto["id_municipio"]))

abt["particao"] = "treino"
abt.iloc[idx_resto, abt.columns.get_loc("particao")] = "teste"
abt.iloc[idx_resto[idx_val], abt.columns.get_loc("particao")] = "validacao"

print("Distribuição das partições:")
print(abt.groupby("particao")
      .agg(alunos=("alvo", "size"),
           municipios=("id_municipio", "nunique"),
           taxa_alfabetizacao=("alvo", lambda s: f"{100 * s.mean():.1f}%"))
      .to_string())

Distribuição das partições:
            alunos  municipios taxa_alfabetizacao
particao                                         
teste       311794        1104              60.8%
treino     1156013        3310              59.4%
validacao   384045        1103              60.0%


In [215]:
# --- 6.2 Conferir que a partição é honesta ---
print("Verificações da partição")
print("=" * 58)

# 1. Nenhum município aparece em mais de uma partição
por_municipio = abt.groupby("id_municipio")["particao"].nunique()
vazados = (por_municipio > 1).sum()
print(f"1. Municípios em mais de uma partição: {vazados}  (esperado: 0)")

# 2. Todas as observações foram atribuídas
sem_particao = abt["particao"].isna().sum()
print(f"2. Observações sem partição: {sem_particao}  (esperado: 0)")

# 3. A variável resposta tem distribuição semelhante entre as partições
taxas = abt.groupby("particao")["alvo"].mean() * 100
amplitude = taxas.max() - taxas.min()
print(f"3. Taxa de alfabetização por partição: "
      f"{', '.join(f'{p} {t:.1f}%' for p, t in taxas.items())}")
print(f"   amplitude entre partições: {amplitude:.1f} pp "
      f"(diferenças pequenas são esperadas: a partição é por município)")

# 4. A cobertura de contexto é semelhante entre as partições
cobertura = abt.groupby("particao")["mun_taxa_ant"].apply(
    lambda s: 100 * s.notna().mean())
print("4. Alunos com contexto por partição: "
      f"{', '.join(f'{p} {c:.1f}%' for p, c in cobertura.items())}")

print("=" * 58)
if vazados or sem_particao:
    raise RuntimeError("Partição reprovada nas verificações acima.")
print("Partição aprovada: nenhum município atravessa as fronteiras.")

Verificações da partição
1. Municípios em mais de uma partição: 0  (esperado: 0)
2. Observações sem partição: 0  (esperado: 0)
3. Taxa de alfabetização por partição: teste 60.8%, treino 59.4%, validacao 60.0%
   amplitude entre partições: 1.4 pp (diferenças pequenas são esperadas: a partição é por município)
4. Alunos com contexto por partição: teste 90.7%, treino 90.2%, validacao 85.2%
Partição aprovada: nenhum município atravessa as fronteiras.


## 7. Gravação da tabela analítica

**Passos desta seção:** (7.1) selecionar as colunas finais e gravar a tabela no data lake; (7.2) reconciliar a gravação e registrar o dicionário de variáveis.

🎓 **Conceito, onde vive a tabela analítica:** o repositório versiona código, e os dados vivem no lake, princípio herdado da fase anterior. A tabela analítica é gravada em uma área própria (`ml/`), separada das camadas do medalhão, porque não é um produto de dados de negócio, e sim um artefato de modelagem, com público e ciclo de vida próprios. As etapas seguintes leem essa tabela, e não voltam às camadas originais.

📌 **O que a tabela carrega, e por quê:**

| Grupo | Colunas | Papel |
|---|---|---|
| Chaves | `ano`, `id_municipio` | rastreabilidade e agregação dos resultados |
| Explicativas | `rede_nome` e as variáveis de contexto da rede e do município | entram no modelo |
| Resposta | `alvo` | o que se quer prever |
| Ponderação | `peso_aluno` | disponível para o treinamento ponderado, não é atributo |
| Partição | `particao` | preserva a separação entre treino, validação e teste em todas as etapas |

📌 **O dicionário de dados** (`reports/dicionario_dados.csv`) descreve cada coluna: bloco, hipótese a que responde, fonte, referência temporal, natureza, fórmula e a **regra de negócio**, isto é, a unidade em que o valor está expresso e como ler o campo. Uma trava impede gravar o dicionário se alguma variável da tabela não tiver essa descrição.

In [216]:
# --- 7.1 Selecionar as colunas finais e gravar no lake ---
from datetime import datetime, timezone

COLUNAS_ABT = CHAVES + FEATURES + [RESPOSTA, "peso_aluno", "particao"]
df_abt = abt[COLUNAS_ABT].copy()

print(f"Tabela analítica: {len(df_abt):,} linhas x {len(df_abt.columns)} colunas")
print(f"  chaves: {CHAVES}   resposta: {RESPOSTA}")
for familia, variaveis in FAMILIAS.items():
    print(f"  {familia:<26} {len(variaveis):>2} variáveis")
print()

garantir_credencial()
momento = datetime.now(timezone.utc)
df_abt["_processing_timestamp"] = momento.isoformat()
destino = (f"gs://{BUCKET_LAKE}/ml/abt_alfabetizacao/"
           f"data_processamento={momento:%Y-%m-%d}/abt_alfabetizacao.parquet")
df_abt.to_parquet(destino, index=False,
                  storage_options={"token": credenciais})
print(f"Gravado em: {destino}")

Tabela analítica: 1,851,852 linhas x 59 colunas
  chaves: ['ano', 'id_municipio']   resposta: alvo
  aluno                       1 variáveis
  desempenho anterior        11 variáveis
  território                  2 variáveis
  porte e oferta              6 variáveis
  infraestrutura escolar     11 variáveis
  profissionais de apoio      3 variáveis
  contexto socioeconômico    20 variáveis

Gravado em: gs://tech-challenge-fase2-lake-rm373453/ml/abt_alfabetizacao/data_processamento=2026-09-08/abt_alfabetizacao.parquet


In [ ]:
# --- 7.2 Reconciliar a gravação e publicar o dicionário de dados ---
garantir_credencial()  # sessões longas: o token expira em cerca de uma hora
relido = pd.read_parquet(destino, storage_options={"token": credenciais})
status = "OK" if len(relido) == len(df_abt) else "DIVERGIU"
print(f"Reconciliação: gravado {len(df_abt):,} | relido {len(relido):,}  {status}")
print()

# Metadados de cada variável: bloco temático, fonte, referência temporal,
# natureza (medida direta, derivada de outras, ou aproximação de um conceito
# que não se mede diretamente) e a fórmula, quando houver.
METADADOS_VARIAVEIS = [
    # (variavel, bloco, fonte, referencia, natureza, descricao, formula)
    ("ano", "chave", "Silver, camada de alunos", str(CICLO_ALVO), "direta",
     "ciclo da avaliação", ""),
    ("id_municipio", "chave", "Silver, camada de alunos", str(CICLO_ALVO), "direta",
     "código IBGE do município", ""),
    ("alvo", "resposta", "Silver, camada de alunos", str(CICLO_ALVO), "derivada",
     "1 se o aluno foi classificado como alfabetizado", "proficiência >= 743"),
    ("peso_aluno", "ponderação", "Silver, camada de alunos", str(CICLO_ALVO), "direta",
     "peso amostral calibrado pelo INEP, reservado à ponderação", ""),
    ("particao", "controle", "construída neste notebook", str(CICLO_ALVO), "derivada",
     "conjunto de destino: treino, validação ou teste", "sorteio por município"),

    ("rede_nome", "aluno", "Silver, camada de alunos", str(CICLO_ALVO), "direta",
     "rede de ensino que atende o aluno", ""),

    ("rede_taxa_ant", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "direta", "taxa de alfabetização da rede do aluno no município", ""),
    ("rede_media_portugues_ant", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "direta", "proficiência média em português da rede", ""),
    ("uf_rede_taxa_ant", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "derivada", "benchmark: mediana da mesma rede entre os municípios da UF",
     "mediana por UF e rede"),
    ("rede_vs_uf", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "derivada", "posição da rede diante do padrão do seu estado",
     "rede_taxa_ant - uf_rede_taxa_ant"),
    ("rede_vs_municipio", "desempenho anterior", "Silver, indicador municipal por rede",
     str(CICLO_ANTERIOR), "derivada", "posição da rede diante do seu município",
     "rede_taxa_ant - mun_taxa_ant"),
    ("mun_taxa_ant", "desempenho anterior", "Gold, indicador municipal",
     str(CICLO_ANTERIOR), "direta", "taxa de alfabetização do município na rede pública", ""),
    ("mun_participacao_ant", "desempenho anterior", "Gold, indicador municipal",
     str(CICLO_ANTERIOR), "direta", "percentual de alunos presentes na avaliação anterior", ""),
    ("mun_taxa_ajustada_ant", "desempenho anterior", "Gold, indicador municipal",
     str(CICLO_ANTERIOR), "derivada", "taxa com ausentes contados como não alfabetizados",
     "mun_taxa_ant * mun_participacao_ant / 100"),
    ("mun_alunos_ant", "desempenho anterior", "Gold, indicador municipal",
     str(CICLO_ANTERIOR), "direta", "alunos presentes no município no ciclo anterior", ""),
    ("mun_meta_ciclo", "desempenho anterior", "Gold, metas pactuadas",
     f"{CICLO_ALVO}, pactuada previamente", "direta",
     "meta de alfabetização pactuada para o ciclo corrente", ""),
    ("mun_gap_meta", "desempenho anterior", "Gold, metas pactuadas",
     f"{CICLO_ANTERIOR} e {CICLO_ALVO}", "derivada", "esforço requerido para atingir a meta",
     "mun_meta_ciclo - mun_taxa_ant"),

    ("sigla_uf", "território", "IBGE, Censo Demográfico", "2022", "direta",
     "unidade da federação", ""),
    ("nome_regiao", "território", "IBGE, código do município", "estável", "derivada",
     "região do país", "primeiro dígito do código IBGE do município"),

    ("rede_porte_atual", "porte e oferta", "Silver, camada de alunos",
     f"{CICLO_ALVO}, cadastro prévio", "derivada",
     "alunos avaliáveis na rede do aluno no município", "contagem de matriculados"),
    ("mun_porte_atual", "porte e oferta", "Silver, camada de alunos",
     f"{CICLO_ALVO}, cadastro prévio", "derivada",
     "alunos avaliáveis no município", "contagem de matriculados"),
    ("rede_peso_no_municipio", "porte e oferta", "Silver, camada de alunos",
     str(CICLO_ALVO), "derivada", "fração dos alunos do município atendida pela rede",
     "rede_porte_atual / mun_porte_atual"),
    ("esc_quantidade", "porte e oferta", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas em atividade da rede no município", ""),
    ("esc_alunos_por_sala", "porte e oferta", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "proxy", "densidade física da rede: alunos por sala em uso, na falta de metragem",
     "matrícula total da rede / salas em uso"),
    ("turma_media_alunos", "porte e oferta", "INEP, Censo Escolar, turmas",
     str(CICLO_ANTERIOR), "direta", "tamanho médio da turma do 2º ano, a série avaliada", ""),

    ("esc_pct_rural", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "derivada", "alunos do 2º ano que estudam em escolas de zona rural",
     "matriculas do 2º ano em escolas rurais / matriculas do 2º ano"),
    ("esc_pct_alunos_zona_rural", "infraestrutura", "INEP, Censo Escolar",
     str(CICLO_ANTERIOR), "proxy",
     "alunos que residem em zona rural, aproximação da distância vivida até a escola",
     "matriculas com residência rural / total de matriculas"),
    ("esc_pct_agua_rede", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com abastecimento de água pela rede pública", ""),
    ("esc_pct_esgoto_rede", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com esgotamento sanitário pela rede pública", ""),
    ("esc_pct_energia_rede", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com energia elétrica da rede pública", ""),
    ("esc_pct_internet", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com acesso à internet", ""),
    ("esc_pct_biblioteca_ou_sala_leitura", "infraestrutura", "INEP, Censo Escolar",
     str(CICLO_ANTERIOR), "direta", "escolas com biblioteca ou sala de leitura", ""),
    ("esc_pct_lab_informatica", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com laboratório de informática", ""),
    ("esc_pct_quadra", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com quadra de esportes", ""),
    ("esc_pct_alimentacao", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas que oferecem alimentação aos alunos", ""),
    ("esc_pct_transporte", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "proxy", "dependência de transporte escolar, aproximação da distância entre aluno e escola",
     "alunos transportados / matrícula total da rede"),
    ("esc_pct_area_diferenciada", "infraestrutura", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "derivada", "escolas em terra indígena, quilombo, assentamento ou comunidade tradicional",
     "escolas com código de área diferenciada diferente de zero / escolas em atividade"),

    ("esc_pct_coordenador", "profissionais", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com coordenador pedagógico", ""),
    ("esc_pct_psicologo", "profissionais", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com psicólogo", ""),
    ("esc_pct_assistente_social", "profissionais", "INEP, Censo Escolar", str(CICLO_ANTERIOR),
     "direta", "escolas com assistente social", ""),

    ("esc_pct_docentes_formacao_adequada", "corpo docente e jornada",
     "INEP, Indicadores Educacionais", str(ANO_INDICADORES), "direta",
     "disciplinas dos anos iniciais dadas por professor com a formação adequada", ""),
    ("esc_pct_docentes_baixa_regularidade", "corpo docente e jornada",
     "INEP, Indicadores Educacionais", str(ANO_INDICADORES), "derivada",
     "professores que permaneceram pouco tempo na mesma escola nos últimos anos",
     "faixa baixa + faixa média-baixa do indicador de regularidade docente"),
    ("esc_pct_docentes_alto_esforco", "corpo docente e jornada",
     "INEP, Indicadores Educacionais", str(ANO_INDICADORES), "derivada",
     "professores dos anos iniciais nos dois níveis mais altos de esforço",
     "nível 5 + nível 6 do indicador de esforço docente"),
    ("esc_alunos_por_docente", "corpo docente e jornada", "INEP, Censo Escolar",
     str(CICLO_ANTERIOR), "derivada", "alunos dos anos iniciais por professor dos anos iniciais",
     "matrículas dos anos iniciais / docentes dos anos iniciais"),
    ("esc_horas_aula_diarias", "corpo docente e jornada", "INEP, Indicadores Educacionais",
     str(ANO_INDICADORES), "direta", "média de horas-aula diárias nos anos iniciais", ""),

    ("mun_pib_per_capita", "socioeconômico", "IBGE, PIB municipal", str(ANO_PIB),
     "derivada", "riqueza produzida por habitante", "PIB / população"),
    ("mun_pct_agropecuaria", "socioeconômico", "IBGE, PIB municipal", str(ANO_PIB),
     "proxy", "peso da agropecuária, aproximação do caráter rural da economia",
     "valor adicionado agropecuário / valor adicionado total"),
    ("mun_pct_servicos", "socioeconômico", "IBGE, PIB municipal", str(ANO_PIB),
     "proxy", "peso dos serviços, aproximação do caráter urbano da economia",
     "valor adicionado de serviços / valor adicionado total"),
    ("mun_populacao", "socioeconômico", "IBGE, estimativa populacional", str(ANO_PIB),
     "direta", "população do município", ""),
    ("mun_idhm", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "índice de desenvolvimento humano municipal", ""),
    ("mun_idhm_educacao", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "dimensão educação do IDHM", ""),
    ("mun_idhm_renda", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "dimensão renda do IDHM", ""),
    ("mun_renda_per_capita", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "renda domiciliar por habitante", ""),
    ("mun_gini", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano", "2010",
     "direta", "desigualdade na distribuição da renda", ""),
    ("mun_alfabetizacao_adulta", "socioeconômico", "IBGE, Censo Demográfico",
     "2022", "proxy",
     "alfabetização entre pessoas de 15 anos ou mais, aproximação do capital cultural do domicílio", ""),
    ("mun_densidade_demografica", "socioeconômico", "IBGE, Censo Demográfico",
     "2022", "derivada", "habitantes por quilômetro quadrado",
     "população / área"),
    ("mun_pessoas_por_domicilio", "socioeconômico", "IBGE, Censo Demográfico",
     "2022", "derivada", "tamanho médio do domicílio",
     "população / domicílios"),
    ("mun_idade_mediana", "socioeconômico", "IBGE, Censo Demográfico",
     "2022", "direta", "idade mediana da população", ""),
    ("mun_pct_esgoto_adequado", "socioeconômico", "IBGE, Censo Demográfico",
     "2022", "direta",
     "população em domicílio com esgotamento por rede ou fossa ligada à rede", ""),
    ("mun_pct_agua_rede_geral", "socioeconômico", "IBGE, Censo Demográfico",
     "2022", "direta",
     "população em domicílio ligado à rede geral de água e que a usa como forma principal", ""),
    ("mun_expectativa_estudo", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano",
     "2010", "direta", "expectativa de anos de estudo", ""),
    ("mun_ivs", "socioeconômico", "IPEA, Atlas da Violência", "2010", "direta",
     "índice de vulnerabilidade social", ""),
    ("mun_ivs_infraestrutura", "socioeconômico", "IPEA, Atlas da Violência", "2010",
     "direta", "vulnerabilidade de infraestrutura urbana", ""),
    ("mun_ivs_capital_humano", "socioeconômico", "IPEA, Atlas da Violência", "2010",
     "direta", "vulnerabilidade de capital humano", ""),
    ("mun_taxa_homicidio", "socioeconômico", "DataSUS, Sistema de Informação sobre Mortalidade",
     str(ANO_VIOLENCIA), "proxy", "óbitos por agressão por cem mil habitantes, aproximação da exposição à violência",
     "óbitos CID X85 a Y09 * 100000 / população"),
    ("esc_inse_medio", "socioeconômico", "INEP, Indicador de Nível Socioeconômico",
     str(ANO_INSE), "direta", "nível socioeconômico médio das famílias atendidas pela rede", ""),
    ("mun_pct_vulner_deslocamento_1h", "socioeconômico", "PNUD, Atlas do Desenvolvimento Humano",
     "2010", "proxy",
     "pessoas pobres que gastam mais de uma hora até o trabalho, aproximação das distâncias no território", ""),

    ("mun_pct_maes_8_anos_estudo", "primeira infância", "Ministério da Saúde, SINASC",
     f"nascidos em {ANOS_NASCIMENTO[0]} e {ANOS_NASCIMENTO[1]}", "derivada",
     "nascidos vivos cuja mãe tinha 8 anos de estudo ou mais",
     "mães com 8 anos de estudo ou mais / mães com escolaridade informada"),
    ("mun_pct_maes_adolescentes", "primeira infância", "Ministério da Saúde, SINASC",
     f"nascidos em {ANOS_NASCIMENTO[0]} e {ANOS_NASCIMENTO[1]}", "derivada",
     "nascidos vivos de mãe com menos de 20 anos", "mães de 10 a 19 anos / mães de 10 a 60 anos"),
    ("mun_indice_pre_escola", "primeira infância", "INEP, Censo Escolar", str(ANO_PRE_ESCOLA),
     "derivada", "oferta de pré-escola diante dos anos iniciais, todas as redes do município",
     "(matrículas da pré-escola / 2) / (matrículas dos anos iniciais / 5)"),
]

COLUNAS_META = ["variavel", "bloco", "fonte", "referencia", "natureza",
                "descricao", "formula"]

# Regra de negócio de cada variável: a unidade em que o valor está expresso e
# como ler o campo. É a parte do dicionário escrita para quem vai usar a base
# sem ter participado da sua construção. As variáveis de rede e de município
# repetem o mesmo valor para todos os alunos da mesma rede no mesmo município.
LEITURA = {
    # chaves, resposta e controle
    "ano": ("ano",
        "Ano do ciclo avaliado. Vale 2024 em todas as linhas; serve para rastrear a origem e não entra no modelo."),
    "id_municipio": ("código IBGE de 7 dígitos",
        "Município em que o aluno estuda. O primeiro dígito indica a região e os dois primeiros, a UF (35 é São Paulo). É a chave de junção com as fontes externas e o critério da partição; não entra no modelo como número."),
    "alvo": ("0 ou 1",
        "1 quando a proficiência do aluno atingiu 743 pontos na escala do SAEB, ponto de corte que o INEP adota para considerar a criança alfabetizada; 0 caso contrário. A média do campo num grupo de alunos é a taxa de alfabetização desse grupo."),
    "peso_aluno": ("peso amostral",
        "Quantos alunos da população cada aluno avaliado representa, segundo a calibração do INEP. Um peso de 1,2 significa que o aluno vale por 1,2 aluno no cálculo da taxa oficial. Serve para ponderar e nunca como atributo."),
    "particao": ("categoria: treino, validacao, teste",
        "Conjunto a que a linha pertence. Todos os alunos de um município estão no mesmo conjunto, para que o teste seja feito com municípios que o modelo não viu."),
    "rede_nome": ("categoria: Estadual, Municipal, Privada",
        "Rede que mantém a escola do aluno. É o único atributo que distingue alunos do mesmo município: as variáveis de rede assumem o valor da rede indicada aqui."),

    # desempenho anterior
    "rede_taxa_ant": ("% de alunos, de 0 a 100",
        "Percentual de alunos da mesma rede, no mesmo município, considerados alfabetizados em 2023. Um valor de 62 significa que 62 de cada 100 alunos daquela rede atingiram o ponto de corte no ano anterior."),
    "rede_media_portugues_ant": ("pontos na escala do SAEB",
        "Proficiência média dos alunos da rede no município em 2023. Complementa a taxa: duas redes com a mesma taxa podem ter médias diferentes, conforme a distância dos seus alunos em relação ao ponto de corte de 743."),
    "uf_rede_taxa_ant": ("% de alunos, de 0 a 100",
        "Mediana da taxa de 2023 entre as redes do mesmo tipo (estadual ou municipal) nos municípios da mesma UF. É o padrão de referência: diz o que é típico para aquela rede naquele estado."),
    "rede_vs_uf": ("pontos percentuais",
        "Taxa da rede do aluno menos a mediana da mesma rede no estado. Positivo indica rede acima do padrão estadual; um valor de -8 significa que a rede ficou 8 pontos abaixo da mediana das redes equivalentes do estado."),
    "rede_vs_municipio": ("pontos percentuais",
        "Taxa da rede do aluno menos a taxa do município como um todo. Positivo indica que o aluno estuda na rede de melhor desempenho do município; perto de zero, que a rede do aluno é a predominante ou que as redes se equivalem."),
    "mun_taxa_ant": ("% de alunos, de 0 a 100",
        "Taxa de alfabetização do município em 2023, somando as redes públicas, conforme o indicador oficial do INEP. Descreve o território como um todo, independentemente da rede do aluno."),
    "mun_participacao_ant": ("% de alunos, de 0 a 100",
        "Percentual dos alunos previstos que compareceram à avaliação de 2023. Participação baixa torna a taxa menos representativa, porque os ausentes podem ter perfil diferente dos presentes."),
    "mun_taxa_ajustada_ant": ("% de alunos, de 0 a 100",
        "Taxa de 2023 recalculada como se todos os ausentes fossem não alfabetizados. É um limite inferior: a distância entre ela e mun_taxa_ant mostra quanto a ausência pode estar elevando o resultado."),
    "mun_alunos_ant": ("alunos",
        "Alunos presentes na avaliação de 2023 no município. Indica o tamanho da amostra por trás da taxa anterior: taxas de municípios com poucos alunos oscilam mais de um ano para outro."),
    "mun_meta_ciclo": ("% de alunos, de 0 a 100",
        "Taxa de alfabetização que o município se comprometeu a atingir em 2024, no pacto federativo. É conhecida antes da prova, por isso não é vazamento. Vazia quando o município não tem meta registrada."),
    "mun_gap_meta": ("pontos percentuais",
        "Meta de 2024 menos a taxa de 2023: quanto o município precisa avançar. Um valor de 10 significa que a meta exige subir 10 pontos em um ano; negativo indica município que já estava acima da meta."),

    # território
    "sigla_uf": ("categoria: UF",
        "Unidade da federação do município. Captura o que é comum aos municípios do estado: política estadual de alfabetização, regime de colaboração entre estado e municípios e história educacional."),
    "nome_regiao": ("categoria: 5 regiões",
        "Grande região do país, derivada do primeiro dígito do código IBGE. É uma versão mais agregada da UF."),

    # porte e oferta
    "rede_porte_atual": ("alunos",
        "Alunos do 2º ano da rede do aluno no município em 2024, presentes e ausentes. Vem do cadastro da avaliação, definido antes da prova."),
    "mun_porte_atual": ("alunos",
        "Alunos do 2º ano do município em 2024, somando as redes. Medida da escala da oferta."),
    "rede_peso_no_municipio": ("fração, de 0 a 1",
        "Parte dos alunos do município que estuda na rede do aluno. Um valor de 0,9 indica a rede predominante; 0,1, uma rede minoritária naquele território."),
    "esc_quantidade": ("escolas",
        "Escolas em atividade da rede no município em 2023, de qualquer etapa. Mede a capilaridade da rede; não se restringe às escolas que oferecem o 2º ano."),
    "esc_alunos_por_sala": ("alunos por sala",
        "Matrícula total da rede dividida pelas salas de aula em uso. Como a sala atende mais de um turno, o valor pode passar de 40 sem indicar superlotação; a leitura é comparativa entre redes. Quanto maior, mais pressionado o espaço físico."),
    "turma_media_alunos": ("alunos por turma",
        "Número médio de alunos nas turmas do 2º ano da rede no município, em 2023. É a medida mais direta de quantas crianças um professor alfabetizador atende ao mesmo tempo."),

    # infraestrutura
    "esc_pct_rural": ("% de alunos, de 0 a 100",
        "Percentual das matrículas do 2º ano da rede que estão em escolas localizadas em zona rural. Diz onde fica a escola, e não onde mora o aluno."),
    "esc_pct_alunos_zona_rural": ("% de alunos, de 0 a 100",
        "Percentual dos alunos da rede, em todas as séries, que residem em zona rural. Diz onde mora o aluno: uma escola urbana pode receber muitos alunos da zona rural, que percorrem distâncias maiores."),
    "esc_pct_agua_rede": ("% de escolas, de 0 a 100",
        "Percentual das escolas em atividade da rede abastecidas pela rede pública de água. Um valor de 70 significa que 70 de cada 100 escolas têm água da rede pública; as demais usam poço, cisterna, rio ou não têm abastecimento."),
    "esc_pct_esgoto_rede": ("% de escolas, de 0 a 100",
        "Percentual das escolas ligadas à rede pública de esgoto. As demais usam fossa ou não têm esgotamento. Em zona rural a fossa é a solução usual, então valor baixo não significa, por si, ausência de saneamento."),
    "esc_pct_energia_rede": ("% de escolas, de 0 a 100",
        "Percentual das escolas atendidas pela rede pública de energia elétrica. As demais usam gerador, energia solar ou não têm energia."),
    "esc_pct_internet": ("% de escolas, de 0 a 100",
        "Percentual das escolas com acesso à internet, para qualquer uso: administrativo, pedagógico ou dos alunos."),
    "esc_pct_biblioteca_ou_sala_leitura": ("% de escolas, de 0 a 100",
        "Percentual das escolas com biblioteca ou sala de leitura. Substitui a contagem só de bibliotecas, que subestimava o acesso a livros nas redes que organizam o acervo em salas de leitura: na rede municipal da capital paulista, 6% das escolas têm biblioteca e 50% têm biblioteca ou sala de leitura."),
    "esc_pct_lab_informatica": ("% de escolas, de 0 a 100",
        "Percentual das escolas com laboratório de informática."),
    "esc_pct_quadra": ("% de escolas, de 0 a 100",
        "Percentual das escolas com quadra de esportes, coberta ou descoberta."),
    "esc_pct_alimentacao": ("% de escolas, de 0 a 100",
        "Percentual das escolas que oferecem alimentação aos alunos. É quase universal na rede pública, por força do programa nacional de alimentação escolar; valores baixos merecem suspeita de falha de registro."),
    "esc_pct_transporte": ("% de alunos",
        "Alunos da rede que usam transporte escolar público, divididos pela matrícula total da rede. Um valor de 30 significa que 3 em cada 10 alunos dependem de transporte para chegar à escola, o que indica distância entre casa e escola."),
    "esc_pct_area_diferenciada": ("% de escolas, de 0 a 100",
        "Percentual das escolas da rede localizadas em área diferenciada: terra indígena, área remanescente de quilombo, assentamento, unidade de uso sustentável ou comunidade tradicional. Na maior parte das redes vale zero, e a leitura útil é presença ou ausência."),

    # profissionais de apoio
    "esc_pct_coordenador": ("% de escolas, de 0 a 100",
        "Percentual das escolas com ao menos um coordenador pedagógico, profissional que organiza e acompanha o trabalho dos professores."),
    "esc_pct_psicologo": ("% de escolas, de 0 a 100",
        "Percentual das escolas com psicólogo no quadro. O Censo registra o profissional lotado na escola; o atendimento prestado pela secretaria de educação, de fora da escola, não aparece aqui."),
    "esc_pct_assistente_social": ("% de escolas, de 0 a 100",
        "Percentual das escolas com assistente social no quadro. Vale a mesma ressalva do psicólogo: o atendimento feito pela secretaria de educação não aparece."),

    # corpo docente e jornada
    "esc_pct_docentes_formacao_adequada": ("% de disciplinas, de 0 a 100",
        "Grupo 1 do Indicador de Adequação da Formação Docente do INEP, nos anos iniciais. O INEP classifica cada par formado por professor e disciplina em cinco grupos; o grupo 1 reúne os casos em que o professor tem licenciatura, ou bacharelado com complementação pedagógica, na área da disciplina que leciona. Um valor de 80 significa que 80% das disciplinas dos anos iniciais da rede são dadas por professor com essa formação. Os grupos 2 a 5 reúnem formação em outra área, sem licenciatura ou sem curso superior."),
    "esc_pct_docentes_baixa_regularidade": ("% de professores, de 0 a 100",
        "Indicador de Regularidade do Docente do INEP, que observa a permanência de cada professor na mesma escola ao longo dos últimos anos e o classifica em quatro faixas: baixa, média-baixa, média-alta e alta. O campo soma as duas faixas inferiores: um valor de 40 significa que 40% dos professores da rede trocaram de escola com frequência ou chegaram há pouco. Refere-se a todas as etapas, e não só aos anos iniciais."),
    "esc_pct_docentes_alto_esforco": ("% de professores, de 0 a 100",
        "Indicador de Esforço Docente do INEP, nos anos iniciais. Ele combina o número de alunos, turnos, escolas e etapas atendidos por professor e o classifica em seis níveis, do menor (1) ao maior esforço (6). O campo soma os níveis 5 e 6: um valor de 10 significa que 10% dos professores da rede atendem muitos alunos, em mais de um turno e, em geral, em mais de uma escola ou etapa."),
    "esc_alunos_por_docente": ("alunos por professor",
        "Matrículas dos anos iniciais divididas pelos professores dos anos iniciais, somando as escolas em atividade da rede. O professor que leciona em duas escolas é contado nas duas, o que puxa o valor para baixo onde isso é comum. Quanto maior, mais alunos por professor."),
    "esc_horas_aula_diarias": ("horas por dia",
        "Média de horas-aula diárias dos alunos dos anos iniciais na rede. O mínimo legal é de 4 horas, e 7 horas ou mais caracterizam o tempo integral. Um valor de 4,5 indica rede quase toda em turno parcial."),

    # socioeconômico
    "mun_pib_per_capita": ("R$ por habitante, valores de 2021",
        "PIB do município dividido pela população. Mede a riqueza produzida no território, e não a renda das famílias: municípios pequenos com mineração, petróleo ou grande indústria têm PIB per capita alto sem que a população seja rica."),
    "mun_pct_agropecuaria": ("% do valor adicionado, de 0 a 100",
        "Parte do valor adicionado da economia do município que vem da agropecuária. Um valor de 40 significa que 40% do que o município produz vem do campo; valores altos indicam economia rural."),
    "mun_pct_servicos": ("% do valor adicionado, de 0 a 100",
        "Parte do valor adicionado que vem dos serviços, sem contar a administração pública. Valores altos indicam economia urbana."),
    "mun_populacao": ("habitantes",
        "População estimada pelo IBGE para 2021."),
    "mun_idhm": ("índice de 0 a 1",
        "Índice de Desenvolvimento Humano Municipal de 2010, que combina longevidade, educação e renda. Faixas do PNUD: até 0,499 muito baixo; 0,500 a 0,599 baixo; 0,600 a 0,699 médio; 0,700 a 0,799 alto; 0,800 ou mais muito alto."),
    "mun_idhm_educacao": ("índice de 0 a 1",
        "Dimensão educação do IDHM de 2010, que combina a escolaridade dos adultos (peso 1) e o fluxo escolar de crianças e jovens (peso 2). Mesmas faixas do IDHM."),
    "mun_idhm_renda": ("índice de 0 a 1",
        "Dimensão renda do IDHM de 2010, calculada a partir da renda por pessoa em escala logarítmica: o mesmo ganho de renda pesa mais nos municípios pobres."),
    "mun_renda_per_capita": ("R$ por mês, valores de 2010",
        "Renda média mensal por pessoa nos domicílios do município. Ao contrário do PIB per capita, mede o que as famílias efetivamente recebem."),
    "mun_gini": ("índice de 0 a 1",
        "Desigualdade da renda entre as pessoas do município, em 2010: 0 seria todos com a mesma renda e 1, toda a renda concentrada em uma pessoa. Quanto maior, mais desigual."),
    "mun_alfabetizacao_adulta": ("% das pessoas de 15 anos ou mais",
        "Percentual das pessoas de 15 anos ou mais que sabem ler e escrever um bilhete simples, critério do Censo de 2022. Aproxima a presença de adultos leitores no domicílio da criança."),
    "mun_densidade_demografica": ("habitantes por km²",
        "População de 2022 dividida pela área do município. Valores baixos indicam população dispersa e, portanto, distâncias maiores até a escola. A distribuição é muito assimétrica: capitais passam de mil habitantes por km², e municípios da Amazônia ficam abaixo de um."),
    "mun_pessoas_por_domicilio": ("pessoas por domicílio",
        "População dividida pelo número de domicílios, em 2022. Valores maiores indicam famílias maiores, que repartem o tempo e a renda dos adultos entre mais pessoas."),
    "mun_idade_mediana": ("anos",
        "Idade que divide a população do município ao meio, em 2022. Valores baixos indicam população jovem, com mais crianças por adulto; valores altos, população envelhecida."),
    "mun_pct_esgoto_adequado": ("% da população, de 0 a 100",
        "Percentual dos moradores cujo domicílio despeja o esgoto na rede geral, na rede pluvial ou em fossa ligada à rede, em 2022. O restante usa fossa não ligada à rede, vala, rio ou não tem banheiro."),
    "mun_pct_agua_rede_geral": ("% da população, de 0 a 100",
        "Percentual dos moradores cujo domicílio está ligado à rede geral de água e a usa como forma principal de abastecimento, em 2022."),
    "mun_expectativa_estudo": ("anos",
        "Anos de estudo que uma criança que entra na escola deve completar até os 18 anos, se os padrões de 2010 se mantiverem. Resume a capacidade do sistema escolar do município de manter os alunos estudando."),
    "mun_ivs": ("índice de 0 a 1",
        "Índice de Vulnerabilidade Social do IPEA, de 2010, média de três dimensões: infraestrutura urbana, capital humano e renda e trabalho. Quanto maior, mais vulnerável. Faixas do IPEA: até 0,200 muito baixa; 0,201 a 0,300 baixa; 0,301 a 0,400 média; 0,401 a 0,500 alta; acima de 0,500 muito alta."),
    "mun_ivs_infraestrutura": ("índice de 0 a 1",
        "Dimensão de infraestrutura urbana do IVS: saneamento, coleta de lixo e tempo de deslocamento até o trabalho da população pobre. Quanto maior, pior."),
    "mun_ivs_capital_humano": ("índice de 0 a 1",
        "Dimensão de capital humano do IVS: mortalidade infantil, crianças fora da escola, maternidade na adolescência, analfabetismo e baixa escolaridade dos adultos do domicílio. Quanto maior, pior."),
    "mun_taxa_homicidio": ("óbitos por 100 mil habitantes",
        "Óbitos por agressão (códigos X85 a Y09 da classificação internacional de doenças) registrados em 2019, por cem mil habitantes. Em município pequeno, um único óbito produz taxa alta, então o valor oscila muito. Município sem registro recebe zero."),
    "esc_inse_medio": ("escala do INEP, valores entre 3,4 e 6,1 na base",
        "Indicador de Nível Socioeconômico médio dos alunos da rede no município, calculado pelo INEP em 2021 com o questionário do SAEB, que pergunta sobre a escolaridade dos pais, a renda e os bens do domicílio. Quanto maior, melhor a condição das famílias. É a única medida socioeconômica no grão da rede: descreve as famílias que a rede atende, e não o município. Zero na fonte foi tratado como ausente."),
    "mun_pct_vulner_deslocamento_1h": ("% de pessoas, de 0 a 100",
        "Pessoas ocupadas em domicílio vulnerável à pobreza (renda por pessoa abaixo de meio salário mínimo de 2010) que gastam mais de uma hora no trajeto até o trabalho, segundo o Atlas do Desenvolvimento Humano. Descreve adultos, e não crianças: entra como aproximação das distâncias e da precariedade do transporte no território. É um dos componentes de mun_ivs_infraestrutura."),

    # primeira infância
    "mun_pct_maes_8_anos_estudo": ("% dos nascidos vivos, de 0 a 100",
        "Entre os nascidos de 2016 e 2017 de mães residentes no município, percentual cuja mãe tinha 8 anos de estudo ou mais no parto, isto é, ao menos o ensino fundamental completo. É a geração que chega ao 2º ano em 2024. Nascimentos com escolaridade ignorada ficam fora do denominador."),
    "mun_pct_maes_adolescentes": ("% dos nascidos vivos, de 0 a 100",
        "Entre os nascidos de 2016 e 2017 de mães residentes no município, percentual cuja mãe tinha de 10 a 19 anos no parto. Um valor de 20 significa que 1 em cada 5 crianças da coorte avaliada nasceu de mãe adolescente."),
    "mun_indice_pre_escola": ("índice, 1 indica equilíbrio",
        "Matrículas na pré-escola por ano de idade divididas pelas matrículas dos anos iniciais por série, somando todas as redes do município em 2022. A pré-escola dura dois anos e os anos iniciais, cinco; dividir cada um pela sua duração torna os dois comparáveis. Valor 1 indica tantas crianças em cada ano da pré-escola quanto em cada série dos anos iniciais; abaixo de 1, que parte das crianças não passou pela pré-escola no município."),
}

# Cada variavel foi eleita para responder a uma hipotese, documentada em
# docs/hipoteses.md. O mapeamento abaixo e a ponte entre o dicionario e o
# raciocinio: e ele que organiza a analise exploratoria.
HIPOTESES = {
    "H1 deslocamento e desgaste": [
        "esc_pct_alunos_zona_rural", "esc_pct_rural", "esc_pct_transporte",
        "mun_pct_agropecuaria", "mun_pct_servicos",
        "mun_densidade_demografica", "mun_pct_vulner_deslocamento_1h"],
    "H2 infraestrutura basica": [
        "esc_pct_agua_rede", "esc_pct_esgoto_rede", "esc_pct_energia_rede",
        "esc_pct_alimentacao"],
    "H3 recursos pedagogicos": [
        "esc_pct_biblioteca_ou_sala_leitura", "esc_pct_lab_informatica", "esc_pct_quadra",
        "esc_pct_internet"],
    "H4 densidade da sala": [
        "turma_media_alunos", "esc_alunos_por_sala", "esc_alunos_por_docente"],
    "H5 suporte especializado": [
        "esc_pct_coordenador", "esc_pct_psicologo", "esc_pct_assistente_social"],
    "H6 capital cultural": [
        "mun_alfabetizacao_adulta", "mun_expectativa_estudo", "mun_idhm_educacao",
        "mun_pessoas_por_domicilio", "mun_pct_maes_8_anos_estudo"],
    "H7 vulnerabilidade e violencia": [
        "mun_ivs", "mun_ivs_infraestrutura", "mun_ivs_capital_humano",
        "mun_taxa_homicidio", "mun_gini", "mun_pct_esgoto_adequado",
        "mun_pct_agua_rede_geral"],
    "H8 inercia territorial": [
        "rede_taxa_ant", "mun_taxa_ant", "rede_media_portugues_ant",
        "uf_rede_taxa_ant", "rede_vs_uf", "rede_vs_municipio",
        "mun_taxa_ajustada_ant", "mun_participacao_ant", "mun_meta_ciclo",
        "mun_gap_meta"],
    "H9 recursos materiais": [
        "mun_pib_per_capita", "mun_renda_per_capita", "mun_idhm_renda",
        "mun_idhm", "mun_idade_mediana", "esc_inse_medio"],
    "H10 corpo docente": [
        "esc_pct_docentes_formacao_adequada", "esc_pct_docentes_baixa_regularidade",
        "esc_pct_docentes_alto_esforco"],
    "H11 tempo na escola": ["esc_horas_aula_diarias"],
    "H12 primeira infancia": ["mun_indice_pre_escola", "mun_pct_maes_adolescentes"],
    "H13 contexto linguistico e cultural": ["esc_pct_area_diferenciada"],
    "controle de escala": [
        "rede_porte_atual", "mun_porte_atual", "rede_peso_no_municipio",
        "esc_quantidade", "mun_populacao", "mun_alunos_ant"],
    "controle territorial": ["sigla_uf", "nome_regiao"],
    "controle do aluno": ["rede_nome"],
    "nao aplicavel": ["ano", "id_municipio", "alvo", "peso_aluno", "particao"],
}
VARIAVEL_HIPOTESE = {v: h for h, vs in HIPOTESES.items() for v in vs}
dicionario = pd.DataFrame(METADADOS_VARIAVEIS, columns=COLUNAS_META)
dicionario["hipotese"] = dicionario["variavel"].map(VARIAVEL_HIPOTESE)
sem_hipotese = dicionario.loc[dicionario["hipotese"].isna(), "variavel"].tolist()
if sem_hipotese:
    raise RuntimeError(
        f"Variaveis sem hipotese declarada: {sem_hipotese}. "
        "Toda variavel precisa responder a uma hipotese de docs/hipoteses.md."
    )

# Trava: toda variável do dicionário precisa da sua regra de negócio
sem_leitura = [v for v in dicionario["variavel"] if v not in LEITURA]
leitura_orfa = [v for v in LEITURA if v not in set(dicionario["variavel"])]
if sem_leitura or leitura_orfa:
    raise RuntimeError(f"Regra de negócio ausente: {sem_leitura}. "
                       f"Regra sem variável: {leitura_orfa}.")
dicionario["unidade"] = dicionario["variavel"].map(lambda v: LEITURA[v][0])
dicionario["como_ler"] = dicionario["variavel"].map(lambda v: LEITURA[v][1])

# Trava: o dicionário e a tabela precisam descrever exatamente o mesmo conjunto
METADADOS_GRAVACAO = ["_processing_timestamp", "data_processamento"]
na_tabela = [c for c in df_abt.columns if c not in METADADOS_GRAVACAO]
sem_descricao = [v for v in na_tabela if v not in set(dicionario["variavel"])]
orfas = [v for v in dicionario["variavel"] if v not in na_tabela]
if sem_descricao or orfas:
    raise RuntimeError(
        f"Dicionário desatualizado. Sem descrição: {sem_descricao}. "
        f"Descrições sem variável: {orfas}. Reexecute as seções 3 a 5."
    )

# Características observadas no dado
dicionario["tipo"] = [str(df_abt[v].dtype) for v in dicionario["variavel"]]
dicionario["preenchimento"] = [
    round(100 * df_abt[v].notna().mean(), 1) for v in dicionario["variavel"]]
dicionario["distintos"] = [df_abt[v].nunique() for v in dicionario["variavel"]]

dicionario = dicionario[["variavel", "bloco", "hipotese", "fonte", "referencia",
                         "natureza", "unidade", "descricao", "como_ler", "formula",
                         "tipo", "preenchimento", "distintos"]]
dicionario.to_csv("../reports/dicionario_dados.csv", index=False,
                  encoding="utf-8")
print(f"{len(dicionario)} variáveis descritas · também salvo em "
      "reports/dicionario_dados.csv")

# O DataFrame fica disponível para inspeção e filtro no próprio notebook.
# Exemplos: dicionario.query("natureza == 'proxy'")
#           dicionario.query("bloco == 'infraestrutura'")
dicionario